<a href="https://colab.research.google.com/github/czcollier/literary-prose-dpo/blob/main/train_literary_prose_gemma2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title install

# Select Runtime -> Change runtime type -> A100 GPU or L4 GPU (High-RAM)
!pip install -q -U "google-genai[aiohttp]" beautifulsoup4 lxml spacy wordfreq pydantic tqdm lemminflect
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets
!pip install -U torchao
!python -m spacy download en_core_web_sm -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
#@title imports

import os
from google.colab import userdata
import asyncio
import json
import random
import re
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from bs4 import BeautifulSoup
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import spacy
from tqdm.asyncio import tqdm
from wordfreq import zipf_frequency
import warnings
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

# Inject secrets into the OS environment
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("API credentials loaded successfully.")

API credentials loaded successfully.


In [ ]:
#@title data preprocessing settings and definitions

MODEL_ID = "gemini-3.8-flash"
TRAINING_MODE = "compose"
WORKDIR = Path("/content/drive/MyDrive/literature_model")
WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATASET_FILE = WORKDIR / f"dpo_lit_{TRAINING_MODE}.jsonl"

nlp = spacy.load("en_core_web_sm", disable=["ner"])
client = genai.Client()

# -----------------------------------------------------------------------------
# Global Dataset Controls & Author Percentage Allocation
# -----------------------------------------------------------------------------
TOTAL_EXAMPLES = 1000
MAX_CONCURRENCY = 18
MAX_ZIPF_SCORE = 3.75


# -----------------------------------------------------------------------------
# Configuration: 1,000 Verified High-Contrast Pairs
# -----------------------------------------------------------------------------

AUTHOR_PERCENTAGES = {
    # F. Scott Fitzgerald (10.0%) - Lyrical sensory texture, rhythmic 20th-c. syntax
    "fitzgerald_the_great_gatsby": 10.0,       # 100 pairs

    # Ernest Hemingway (10.0%) - Stripped concrete nouns, active verbs, zero filler
    "hemingway_the_sun_also_rises": 5.0,       # 50 pairs
    "hemingway_a_farewell_to_arms": 5.0,       # 50 pairs

    # Virginia Woolf (16.0%) - Atmospheric impressionism & sensory cadence
    "woolf_to_the_lighthouse": 8.0,            # 80 pairs
    "woolf_mrs_dalloway": 8.0,                 # 80 pairs

    # Joseph Conrad (24.0%) - Muscular periodic clauses & physical spine
    "conrad_nostromo": 14.0,                   # 140 pairs
    "conrad_the_secret_agent": 8.0,            # 80 pairs
    "conrad_lord_jim": 2.0,                    # 20 pairs

    # Herman Melville (18.0%) - Tactile material nouns & maritime momentum
    "melville_moby_dick": 10.0,                # 100 pairs
    "melville_typee": 8.0,                     # 80 pairs

    # Henry James (14.0%) - Spatial staging & delicate subordination
    "james_the_portrait_of_a_lady": 9.0,       # 90 pairs
    "james_the_ambassadors": 3.0,              # 30 pairs
    "james_the_turn_of_the_screw": 2.0,        # 20 pairs

    # James Joyce (8.0%) - Dubliners: naturalistic restraint, complete sentences
    "joyce_dubliners": 8.0,                    # 80 pairs
}

REPOSITORIES = {
    # F. Scott Fitzgerald
    "fitzgerald_the_great_gatsby": "https://github.com/standardebooks/f-scott-fitzgerald_the-great-gatsby.git",

    # Ernest Hemingway
    "hemingway_the_sun_also_rises": "https://github.com/standardebooks/ernest-hemingway_the-sun-also-rises.git",
    "hemingway_a_farewell_to_arms": "https://github.com/standardebooks/ernest-hemingway_a-farewell-to-arms.git",

    # Virginia Woolf
    "woolf_to_the_lighthouse": "https://github.com/standardebooks/virginia-woolf_to-the-lighthouse.git",
    "woolf_mrs_dalloway": "https://github.com/standardebooks/virginia-woolf_mrs-dalloway.git",

    # Joseph Conrad
    "conrad_nostromo": "https://github.com/standardebooks/joseph-conrad_nostromo.git",
    "conrad_the_secret_agent": "https://github.com/standardebooks/joseph-conrad_the-secret-agent.git",
    "conrad_lord_jim": "https://github.com/standardebooks/joseph-conrad_lord-jim.git",

    # Herman Melville
    "melville_moby_dick": "https://github.com/standardebooks/herman-melville_moby-dick.git",
    "melville_typee": "https://github.com/standardebooks/herman-melville_typee.git",

    # Henry James
    "james_the_portrait_of_a_lady": "https://github.com/standardebooks/henry-james_the-portrait-of-a-lady.git",
    "james_the_ambassadors": "https://github.com/standardebooks/henry-james_the-ambassadors.git",
    "james_the_turn_of_the_screw": "https://github.com/standardebooks/henry-james_the-turn-of-the-screw.git",

    # James Joyce
    "joyce_dubliners": "https://github.com/standardebooks/james-joyce_dubliners.git",
}


def compute_integer_quotas(percentages: Dict[str, float], total: int) -> Dict[str, int]:
    """
    Converts floating-point percentage shares into exact integer sample targets
    using the largest-remainder method (Hamilton-Hare) to prevent rounding drift.
    """
    total_pct = sum(percentages.values())
    if not (99.9 <= total_pct <= 100.1):
        raise ValueError(f"Percentages must sum to 100.0 (current sum: {total_pct:.2f})")

    exact_counts = {repo: (pct / 100.0) * total for repo, pct in percentages.items()}
    quotas = {repo: int(count) for repo, count in exact_counts.items()}

    # Distribute leftover rounding remainder to repositories with highest fractional values
    remainder = total - sum(quotas.values())
    fractions = sorted(
        percentages.keys(),
        key=lambda r: exact_counts[r] - quotas[r],
        reverse=True
    )
    for i in range(remainder):
        quotas[fractions[i]] += 1

    return quotas

In [ ]:
#@title data preprocessing helpers

# -----------------------------------------------------------------------------
# Schema: Continuous Scene Spine (No Fragmented Checklists)
# -----------------------------------------------------------------------------
class SceneSpineDPOPair(BaseModel):
    prompt: str = Field(
        description=(
            "A continuous 2-3 sentence narrative outline in the present tense. "
            "Do NOT use bullet points, brackets, or category labels like 'Setting:' or 'Sensory Inventory:'. "
            "Depict an active subject interacting with a concrete physical instrument, "
            "obstacle, or material in a defined physical space. "
            "Example: 'A clerk in an unheated basement office folds a damp parchment lease, "
            "slips his brass spectacle case into a waistcoat pocket, and hesitates in the soot-thick air.'"
        )
    )
    chosen: str = Field(
        description=(
            "The master stylist excerpt adapted strictly into immediate present tense. "
            "CRITICAL: Must consist of fully formed, grammatically complete periodic sentences "
            "with active finite verbs and clear clause subordination. "
            "NEVER produce verbless nominal catalogs or staccato noun lists. "
            "Preserve rare anchor words and conclude with an explicit full stop."
        )
    )
    rejected: str = Field(
        description=(
            "The exact same scene rewritten in typical clichéd, bloated assistant prose: "
            "Uses abstract summarizing ('a poignant testament to', 'quiet resolve', 'steeped in', 'palpable tension'), "
            "symmetrical rule-of-three descriptors, weak copulas ('was', 'felt', 'seemed'), "
            "and an unearned moralizing or explanatory concluding phrase."
        )
    )

# -----------------------------------------------------------------------------
# Asynchronous Gemini Harvesting
# -----------------------------------------------------------------------------
SYSTEM_INSTRUCTION = """You are an expert computational stylistician creating high-contrast DPO training data.
Your goal is to teach an LLM to produce muscular, periodic, present-tense literary prose with complete syntactic clauses, while completely suppressing the clichéd, moralizing corporate AI assistant voice.

Given a master prose excerpt:
1. 'prompt': Construct a continuous 2-3 sentence present-tense narrative outline ('Scene Spine'). Do NOT use bullet points, headers, or checklists. Describe an active protagonist interacting with a specific physical object or environment.
2. 'chosen': Render the master excerpt into immediate present tense. Enforce complete, periodic sentence structure with finite active verbs. NEVER allow verbless fragment lists or staccato nominal catalogs. Ensure the passage ends on a clean period.
3. 'rejected': Rephrase the scene into the standard AI assistant voice: fill it with abstract emotional commentary ('a testament to', 'poignant reminder', 'steeped in', 'palpable'), symmetrical three-part lists, and an unearned thematic summary at the end."""

async def process_candidate(
    semaphore: asyncio.Semaphore,
    prose: str,
    anchors: List[str]
) -> Optional[Dict[str, str]]:
    user_prompt = f"""Master Excerpt:
"{prose}"

Rare anchors that must be synthesized into 'chosen' and omitted/diluted in 'rejected':
{', '.join(anchors)}

Output strictly valid JSON conforming to the schema."""

    async with semaphore:
        for attempt in range(1, 4):
            try:
                res = await client.aio.models.generate_content(
                    model=MODEL_ID,
                    contents=user_prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_INSTRUCTION,
                        temperature=0.25,
                        response_mime_type="application/json",
                        response_schema=SceneSpineDPOPair,
                    ),
                )
                data = json.loads(res.text)

                # Format into standard boundary delimiter matching your fine-tuning pipeline
                prompt_formatted = f"[SCENE SPINE]\n{data['prompt'].strip()}\n\n[PROSE]\n"
                chosen_formatted = data["chosen"].strip()
                rejected_formatted = data["rejected"].strip()

                return {
                    "prompt": prompt_formatted,
                    "chosen": chosen_formatted,
                    "rejected": rejected_formatted,
                }
            except Exception:
                await asyncio.sleep(1.5 ** attempt)
        return None
# -----------------------------------------------------------------------------
# Extraction, Robust Sentence Filtering, and Zipf Anchoring
# -----------------------------------------------------------------------------
def clone_repo(name: str, url: str) -> Path:
    p = WORKDIR / "repos" / name
    if not p.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", url, str(p)],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    return p


# Suppress BS4 XML parser warning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

def is_grammatically_robust(text: str) -> bool:
    """
    Ensures the paragraph has solid syntactic architecture without
    throwing away paragraphs for minor stylistic flourishes or dialogue dashes.
    """
    # Exclude standalone direct spoken dialogue
    if text.startswith(('"', "'", "“")):
        return False

    doc = nlp(text)
    sentences = [sent for sent in doc.sents if len(sent.text.strip()) > 12]

    # Require at least 2 distinct sentences
    if len(sentences) < 2:
        return False

    valid_sentences = 0
    for sent in sentences:
        has_root_verb = any(
            t.pos_ == "VERB" and t.dep_ == "ROOT" and t.morph.get("VerbForm") != ["Inf"]
            for t in sent
        )
        has_subject = any(t.dep_ in ("nsubj", "nsubjpass", "csubj") for t in sent)

        if has_root_verb and has_subject:
            valid_sentences += 1

    # At least 70% of sentences must be complete periodic clauses (allows occasional em-dash appositives)
    return (valid_sentences / len(sentences)) >= 0.70

def extract_paragraphs_from_repo(repo_path: Path) -> List[str]:
    text_dir = repo_path / "src" / "epub" / "text"
    if not text_dir.exists():
        text_dir = repo_path / "src"

    paragraphs = []
    for fpath in sorted(text_dir.glob("*.xhtml")):
        if any(skip in fpath.name for skip in ["toc", "colophon", "titlepage", "imprint"]):
            continue
        with open(fpath, "r", encoding="utf-8") as f:
            # Parse cleanly as XML to avoid malformed entity warnings
            soup = BeautifulSoup(f.read(), "lxml-xml")

        for p in soup.find_all("p"):
            if p.find_parent(["header", "footer"]) or p.get("epub:type") == "footnote":
                continue

            text = re.sub(r"\s+", " ", p.get_text()).strip()
            word_count = len(text.split())

            # Sweet spot for scene beats (40 to 180 words)
            if 40 <= word_count <= 180:
                if is_grammatically_robust(text):
                    paragraphs.append(text)

            # For longer Conradian paragraphs (>180 words), slice into a 2-3 sentence chunk
            elif word_count > 180:
                doc = nlp(text)
                sents = [s.text.strip() for s in doc.sents]
                # Grab a cohesive 2-3 sentence window
                if len(sents) >= 3:
                    chunk = " ".join(sents[:3])
                    if 45 <= len(chunk.split()) <= 140 and is_grammatically_robust(chunk):
                        paragraphs.append(chunk)

    return paragraphs

def extract_passages_from_repo(repo_path: Path) -> List[str]:
    src_dir = repo_path / "src"
    if not src_dir.exists():
        src_dir = repo_path

    xhtml_files = sorted(src_dir.rglob("*.xhtml"))
    passages = []

    for fpath in xhtml_files:
        fname = fpath.name.lower()
        if any(skip in fname for skip in ["toc", "colophon", "titlepage", "imprint", "halftitle", "uncopyright"]):
            continue

        with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "lxml-xml")

        raw_paras = []
        for p in soup.find_all("p"):
            if p.find_parent(["header", "footer", "blockquote"]) or p.get("epub:type") in ("footnote", "epigraph"):
                continue

            txt = re.sub(r"\s+", " ", p.get_text()).strip()
            if not txt.startswith(('"', "'", "“", "—")) and len(txt.split()) >= 12:
                raw_paras.append(txt)

        i = 0
        while i < len(raw_paras):
            curr = raw_paras[i]
            words = len(curr.split())

            # Case 1: Standalone sweet spot (40 to 160 words)
            if 40 <= words <= 160 and is_grammatically_robust(curr):
                passages.append(curr)
                i += 1
                continue

            # Case 2: Too brief (<40 words) -> try stitching with the next paragraph
            if words < 40 and i + 1 < len(raw_paras):
                combined = curr + " " + raw_paras[i + 1]
                comb_words = len(combined.split())
                if 40 <= comb_words <= 160 and is_grammatically_robust(combined):
                    passages.append(combined)
                    i += 2  # Consumed both successfully
                    continue

            # Case 3: Overly long (>160 words) -> extract cohesive 3-sentence window
            if words > 160:
                doc = nlp(curr)
                sents = [s.text.strip() for s in doc.sents]
                if len(sents) >= 3:
                    chunk = " ".join(sents[:3])
                    chunk_words = len(chunk.split())
                    if 40 <= chunk_words <= 150 and is_grammatically_robust(chunk):
                        passages.append(chunk)

            i += 1

    return passages


def extract_rare_anchors(text: str) -> List[Tuple[str, float]]:
    doc = nlp(text)
    anchors = []
    for token in doc:
        if token.is_stop or token.is_punct or len(token.text) < 4:
            continue
        if token.pos_ in ("ADJ", "NOUN", "VERB"):
            score = zipf_frequency(token.lemma_.lower(), "en")
            # Zipf <= 3.75 captures concrete tactile terms (bulkhead, fog, mast, shroud)
            if "-" in token.text or (0.1 < score <= MAX_ZIPF_SCORE):
                anchors.append((token.text, score))
    return anchors


In [ ]:
#@title check sample counts

quotas = compute_integer_quotas(AUTHOR_PERCENTAGES, TOTAL_EXAMPLES)
print(f"Targeting TOTAL_EXAMPLES = {TOTAL_EXAMPLES}:\n")

for repo_name, url in REPOSITORIES.items():
    r_path = clone_repo(repo_name, url)
    passages = extract_paragraphs_from_repo(r_path)

    valid_count = sum(1 for p in passages if len(extract_rare_anchors(p)) >= 1)
    needed = quotas[repo_name]
    status = "✓ OK" if valid_count >= needed else f"⚠️ DEFICIT (-{needed - valid_count})"
    print(f"  • {repo_name:<30}: Available {valid_count:>4} | Needed {needed:>4} -> {status}")

In [ ]:
#@title run data pipeline

# -----------------------------------------------------------------------------
# Pipeline Controller with Dynamic Percentage Allocator
# -----------------------------------------------------------------------------
async def run_pipeline():
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
    selected_pool: List[Tuple[str, List[str]]] = []

    # Calculate exact integer target counts from percentage weights
    quotas = compute_integer_quotas(AUTHOR_PERCENTAGES, TOTAL_EXAMPLES)

    print(f"Targeting TOTAL_EXAMPLES = {TOTAL_EXAMPLES} across weighted repositories:")
    for repo, count in quotas.items():
        pct = AUTHOR_PERCENTAGES[repo]
        print(f"  • {repo:<28}: {pct:>5.1f}% -> {count:>4} pairs")
    print("-" * 60)

    print("Cloning repositories and extracting grammatically robust candidates...")
    for repo_name, url in REPOSITORIES.items():
        target_count = quotas[repo_name]
        r_path = clone_repo(repo_name, url)
        passages = extract_passages_from_repo(r_path)

        valid_for_repo = []
        for p in passages:
            anchors = extract_rare_anchors(p)
            if len(anchors) >= 1:
                valid_for_repo.append((p, [a[0] for a in anchors[:6]]))

        # Shuffle within author pool to ensure wide chapter representation
        random.seed(42)
        random.shuffle(valid_for_repo)

        if len(valid_for_repo) < target_count:
            print(f"  ⚠️ Warning: {repo_name} extracted only {len(valid_for_repo)} valid passages (needed {target_count}). Using all available.")
            allocated = valid_for_repo
        else:
            allocated = valid_for_repo[:target_count]

        selected_pool.extend(allocated)
        print(f"  ✓ {repo_name:<28}: extracted {len(valid_for_repo):>4} | allocated {len(allocated):>4}")

    # Shuffle the overall pool so training batches see mixed author styles
    random.shuffle(selected_pool)
    print(f"\nFinal randomized queue: {len(selected_pool)} items")
    print(f"Dispatching to {MODEL_ID} under concurrency {MAX_CONCURRENCY}...")

    tasks = [process_candidate(semaphore, p, anc) for p, anc in selected_pool]

    saved_count = 0
    with open(OUTPUT_DATASET_FILE, "w", encoding="utf-8") as f:
        for coro in tqdm.as_completed(tasks, total=len(tasks), desc="Harvesting Pairs"):
            result = await coro
            if result:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")
                saved_count += 1

    print(f"\nDone. Saved {saved_count} quota-weighted pairs to:")
    print(f" -> {OUTPUT_DATASET_FILE}")

# Execute in notebook
await run_pipeline()

In [ ]:
#@title Strip proper nounds from the dataset

import json
import re
from pathlib import Path
from typing import Dict, List, Optional
import spacy
from lemminflect import getInflection

# -----------------------------------------------------------------------------
# 1. Configuration & Paths
# -----------------------------------------------------------------------------
INPUT_JSONL = Path("/content/drive/MyDrive/literature_model/dpo_lit_compose.jsonl")
OUTPUT_JSONL = Path("/content/drive/MyDrive/literature_model/dpo_lit_compose_present_deanonymized.jsonl")

# Direct canonical substitutions for dominant proper names across the 12 source books
CANONICAL_ENTITIES: Dict[str, str] = {
    # The Great Gatsby
    "Gatsby": "the host",
    "Jay Gatsby": "the host",
    "Mr. Gatsby": "the host",
    "Daisy": "the woman",
    "Daisy Buchanan": "the woman",
    "Tom Buchanan": "the husband",
    "Jordan Baker": "the guest",
    "Nick Carraway": "the observer",
    "Myrtle Wilson": "the woman",
    "George Wilson": "the mechanic",
    "Wolfsheim": "the associate",
    "Meyer Wolfsheim": "the associate",
    # The Secret Agent / Lord Jim / Nostromo
    "Verloc": "the proprietor",
    "Mr. Verloc": "the proprietor",
    "Mrs. Verloc": "the wife",
    "Winnie": "the wife",
    "Stevie": "the youth",
    "Chief Inspector Heat": "the inspector",
    "Michaelis": "the companion",
    "Ossipon": "the comrade",
    "Vladimir": "the official",
    "Lord Jim": "the young officer",
    "Jim": "the young sailor",
    "Marlow": "the narrator",
    "Nostromo": "the foreman",
    "Charles Gould": "the administrator",
    "Mrs. Gould": "the lady",
    "Gian' Battista": "the boatman",
    # Moby-Dick / Typee
    "Ahab": "the captain",
    "Captain Ahab": "the captain",
    "Queequeg": "the harpooner",
    "Starbuck": "the first mate",
    "Ishmael": "the traveler",
    "Stubb": "the mate",
    "Flask": "the second mate",
    "Fedallah": "the parsee",
    "Tashtego": "the hunter",
    "Daggoo": "the tall sailor",
    "Pip": "the cabin boy",
    "Pequod": "the vessel",
    "Toby": "the companion",
    "Mehevi": "the chief",
    "Fayaway": "the islander",
    # To the Lighthouse / Mrs Dalloway
    "Mrs. Ramsay": "the matriarch",
    "Mr. Ramsay": "the scholar",
    "Lily Briscoe": "the painter",
    "Charles Tansley": "the disciple",
    "William Bankes": "the elder friend",
    "Paul Rayley": "the young suitor",
    "Minta Doyle": "the young guest",
    "Clarissa Dalloway": "the hostess",
    "Mrs. Dalloway": "the hostess",
    "Richard Dalloway": "the statesman",
    "Peter Walsh": "the traveler",
    "Septimus Warren Smith": "the soldier",
    "Septimus": "the veteran",
    "Rezia": "his wife",
    "Lucrezia": "the young wife",
    "Hugh Whitbread": "the courtier",
    "Lady Bruton": "the patroness",
    "Miss Kilman": "the companion",
    # The Portrait of a Lady / The Ambassadors / The Turn of the Screw
    "Isabel Archer": "the young woman",
    "Isabel": "the young woman",
    "Gilbert Osmond": "the connoisseur",
    "Osmond": "the connoisseur",
    "Madame Merle": "the confidante",
    "Ralph Touchett": "her cousin",
    "Lord Warburton": "the nobleman",
    "Caspar Goodwood": "the suitor",
    "Henrietta Stackpole": "the correspondent",
    "Pansy": "the daughter",
    "Lambert Strether": "the envoy",
    "Strether": "the envoy",
    "Chad Newsome": "the young man",
    "Chad": "the heir",
    "Madame de Vionnet": "the countess",
    "Maria Gostrey": "his guide",
    "Waymarsh": "his companion",
    "Miles": "the boy",
    "Flora": "the child",
    "Mrs. Grose": "the housekeeper",
    "Peter Quint": "the valet",
    "Miss Jessel": "the former governess",
    # Dubliners
    "Father Flynn": "the old priest",
    "Lenehan": "the lounger",
    "Corley": "his associate",
    "Mr. Doran": "the clerk",
    "Bob Doran": "the lodger",
    "Polly Mooney": "the daughter",
    "Little Chandler": "the clerk",
    "Gallaher": "the journalist",
    "Mr. Duffy": "the solitary clerk",
    "James Duffy": "the clerk",
    "Mrs. Sinico": "the lady",
    "Gabriel Conroy": "the writer",
    "Gabriel": "the man",
    "Gretta": "his wife",
    "Miss Ivors": "his colleague",
    "Freddy Malins": "the guest",
}

# Compile entity regex sorted by longest first to avoid partial replacements
SORTED_KEYS = sorted(CANONICAL_ENTITIES.keys(), key=lambda s: len(s), reverse=True)
ENTITY_PATTERN = re.compile(r"\b(" + "|".join(re.escape(k) for k in SORTED_KEYS) + r")\b")

# Auxiliary past-tense mapping for direct lookup
AUX_PAST_TO_PRESENT = {
    "was": "is",
    "were": "are",
    "had": "has",
    "did": "does",
    "could": "can",
    "would": "will",
    "might": "may",
}

# -----------------------------------------------------------------------------
# 2. Linguistic Processing (spaCy + lemminflect)
# -----------------------------------------------------------------------------
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")


def scrub_entities(text: str) -> str:
    """Replaces book-specific character entities with generic literary roles."""
    def _sub(match):
        orig = match.group(0)
        replacement = CANONICAL_ENTITIES.get(orig, orig)
        return replacement

    text = ENTITY_PATTERN.sub(_sub, text)

    # Clean residual courtesy titles that had other non-canonical surnames
    text = re.sub(r"\b(Mr\.|Mrs\.|Miss|Captain|Lord|Lady)\s+[A-Z][a-z]+", "the traveler", text)
    return text


def find_subject_number(verb_token) -> str:
    """Determines if the subject of a finite verb is plural or singular."""
    for child in verb_token.children:
        if child.dep_ in ("nsubj", "nsubjpass"):
            # Check morphology or part of speech
            morph = child.morph.to_dict()
            if morph.get("Number") == "Plur" or child.tag_ in ("NNS", "NNPS"):
                return "Plur"
            if child.lower_ in ("they", "we", "all", "both"):
                return "Plur"
            if child.lower_ in ("i", "you"):
                return "Sing"
            return "Sing"

    # Inherit from conjunction head if coordinated (e.g. 'he walks and [talks]')
    if verb_token.dep_ == "conj" and verb_token.head.pos_ == "VERB":
        return find_subject_number(verb_token.head)

    return "Sing"


def convert_verb_to_present(token) -> Optional[str]:
    """Converts a VBD (past tense) verb token to present tense (VBZ/VBP)."""
    lower = token.text.lower()

    # Direct modal/copula overrides
    if lower in AUX_PAST_TO_PRESENT:
        res = AUX_PAST_TO_PRESENT[lower]
        # Preserve original casing
        if token.text.isupper():
            return res.upper()
        if token.text[0].isupper():
            return res.capitalize()
        return res

    number = find_subject_number(token)
    target_tag = "VBP" if number == "Plur" else "VBZ"

    # Query lemminflect
    infl = getInflection(token.lemma_, tag=target_tag)
    if infl:
        res = infl[0]
        if token.text[0].isupper():
            res = res.capitalize()
        return res

    return None


def transform_to_present_tense(text: str) -> str:
    """Parses text with spaCy and converts finite past-tense verbs (VBD) to present tense."""
    doc = nlp(text)
    token_texts = [t.text_with_ws for t in doc]

    for i, token in enumerate(doc):
        # Only alter main finite verbs and auxiliaries marked as VBD (past tense)
        if token.tag_ == "VBD":
            # Avoid changing gerunds, infinitives, or past participles (VBN)
            pres = convert_verb_to_present(token)
            if pres:
                # Maintain trailing whitespace
                trailing_ws = token.text_with_ws[len(token.text):]
                token_texts[i] = pres + trailing_ws

    result = "".join(token_texts)
    # Tidy minor punctuation/spacing anomalies
    result = re.sub(r"\s+", " ", result).strip()
    return result


def process_entry(entry: Dict[str, str]) -> Dict[str, str]:
    """Normalizes prompt, chosen, and rejected components."""
    # 1. Scrub proper entities across all fields
    prompt_clean = scrub_entities(entry["prompt"])
    chosen_clean = scrub_entities(entry["chosen"])
    rejected_clean = scrub_entities(entry.get("rejected", ""))

    # 2. Transform chosen prose to strict present tense
    chosen_present = transform_to_present_tense(chosen_clean)

    # 3. If prompt spine verbs drifted to past tense, normalize prompt as well
    prompt_present = transform_to_present_tense(prompt_clean)

    return {
        "prompt": prompt_present,
        "chosen": chosen_present,
        "rejected": scrub_entities(rejected_clean),
    }


# -----------------------------------------------------------------------------
# 3. Main Transformation Loop
# -----------------------------------------------------------------------------
def run_dataset_normalization():
    if not INPUT_JSONL.exists():
        print(f"Error: {INPUT_JSONL} does not exist.")
        return

    print(f"Reading dataset: {INPUT_JSONL}")
    with open(INPUT_JSONL, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]

    print(f"Processing {len(records)} records (entity scrubbing + present-tense conjugation)...")
    cleaned_records = []

    for idx, rec in enumerate(records):
        try:
            cleaned = process_entry(rec)
            cleaned_records.append(cleaned)
        except Exception as e:
            print(f"Warning: skipped entry {idx} due to parsing error: {e}")

        if (idx + 1) % 100 == 0:
            print(f"[{idx + 1}/{len(records)}] completed.")

    print(f"\nWriting cleaned dataset to: {OUTPUT_JSONL}")
    with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
        for rec in cleaned_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print("Normalization complete.\n")

    # Inspect before/after verification on a sample
    if cleaned_records:
        sample_orig = records[0]
        sample_clean = cleaned_records[0]
        print("=" * 70)
        print("VERIFICATION SAMPLE (Original Chosen):")
        print(sample_orig["chosen"][:300])
        print("-" * 70)
        print("VERIFICATION SAMPLE (Normalized Chosen):")
        print(sample_clean["chosen"][:300])
        print("=" * 70)


if __name__ == "__main__":
    run_dataset_normalization()

In [ ]:
#@title check random examples

import json
import random
from pathlib import Path

DATASET_PATH = Path("/content/drive/MyDrive/literature_model/dpo_lit_compose_present_deanonymized.jsonl")

def inspect_random_samples(path: Path, n: int = 3):
    if not path.exists():
        print(f"File not found: {path}")
        return

    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    total = len(lines)
    print(f"Total dataset entries: {total}")
    if total == 0:
        return

    indices = random.sample(range(total), min(n, total))

    for idx, sample_idx in enumerate(indices, 1):
        record = json.loads(lines[sample_idx])
        print("\n" + "=" * 80)
        print(f"SAMPLE {idx} (Index {sample_idx})")
        print("=" * 80)

        print("\n[PROMPT / SCENE SPINE]")
        print(record.get("prompt", "").strip())

        print("\n" + "-" * 40)
        print("[CHOSEN (Target Prose)]")
        print(record.get("chosen", "").strip())

        print("\n" + "-" * 40)
        print("[REJECTED (Assistant Baseline)]")
        print(record.get("rejected", "").strip())
        print("=" * 80 + "\n")

inspect_random_samples(DATASET_PATH, n=3)

In [ ]:
#@title run sft

import os
import gc
import json
from pathlib import Path
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# =============================================================================
# 1. CORE PATHS & MODEL IDENTIFIERS
# =============================================================================
BASE_MODEL_ID = "google/gemma-2-9b"
WORKDIR = Path("/content/drive/MyDrive/literature_model")
DATASET_FILE = WORKDIR / "dpo_lit_compose_present_deanonymized.jsonl"
SFT_OUTPUT_DIR = WORKDIR / "gemma_lit_sft_warmup_v2"
DPO_OUTPUT_DIR = WORKDIR / "gemma_lit_dpo_final_v2"

PROMPT_DELIMITER = "\n\n[PROSE]\n"
MAX_SEQ_LENGTH = 768
MAX_PROMPT_LENGTH = 256

# =============================================================================
# 2. QLORA ARCHITECTURE HYPERPARAMETERS
# =============================================================================
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# =============================================================================
# 3. STAGE 1 (SFT WARM-UP) HYPERPARAMETERS
# =============================================================================
SFT_EPOCHS = 2.5
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM_STEPS = 4           # Effective batch size = 4 (~241 optimizer steps)
SFT_LEARNING_RATE = 3.5e-5
SFT_WARMUP_RATIO = 0.08
SFT_LR_SCHEDULER = "cosine"
SFT_OPTIMIZER = "paged_adamw_8bit"
SFT_LOGGING_STEPS = 10

# Environment hygiene
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# =============================================================================
# QUANTIZATION & TOKENIZER INITIALIZATION
# =============================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# =============================================================================
# DATASET LOADING
# =============================================================================
print(f"Reading dataset: {DATASET_FILE}")
records = []
with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} verified training pairs.")
raw_dataset = Dataset.from_list(records)

# =============================================================================
# STAGE 1 EXECUTION: SFT WARM-UP (MASKED LABELS)
# =============================================================================
print("\n" + "=" * 60)
print(f"STAGE 1: SFT WARM-UP ({SFT_EPOCHS} Epoch, LR={SFT_LEARNING_RATE})")
print("=" * 60)

def preprocess_sft_batch(example):
    prompt_text = example["prompt"].strip()
    if not prompt_text.endswith(PROMPT_DELIMITER.strip()):
        prompt_text = prompt_text + PROMPT_DELIMITER
    else:
        prompt_text = prompt_text + "\n"

    target_text = example["chosen"].strip() + tokenizer.eos_token

    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)
    target_ids = tokenizer.encode(target_text, add_special_tokens=False)

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    input_ids = input_ids[:MAX_SEQ_LENGTH]
    labels = labels[:MAX_SEQ_LENGTH]
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
    }

sft_tokenized = raw_dataset.map(preprocess_sft_batch, remove_columns=raw_dataset.column_names)

effective_batch_size = SFT_BATCH_SIZE * SFT_GRAD_ACCUM_STEPS
total_train_steps = int((len(sft_tokenized) / effective_batch_size) * SFT_EPOCHS)
sft_warmup_steps = max(1, int(total_train_steps * SFT_WARMUP_RATIO))

print("Instantiating base model in 4-bit precision...")
base_model_sft = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
base_model_sft = prepare_model_for_kbit_training(base_model_sft)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

peft_sft_model = get_peft_model(base_model_sft, lora_config)
peft_sft_model.print_trainable_parameters()

sft_training_args = TrainingArguments(
    output_dir=str(SFT_OUTPUT_DIR),
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM_STEPS,
    learning_rate=SFT_LEARNING_RATE,
    warmup_steps=sft_warmup_steps,
    lr_scheduler_type=SFT_LR_SCHEDULER,
    logging_steps=SFT_LOGGING_STEPS,
    save_strategy="no",
    bf16=True,
    optim=SFT_OPTIMIZER,
    report_to="none",
    gradient_checkpointing=True,
)

sft_trainer = Trainer(
    model=peft_sft_model,
    args=sft_training_args,
    train_dataset=sft_tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
)

print("Starting SFT warm-up...")
sft_trainer.train()

print(f"Saving SFT adapter checkpoint to: {SFT_OUTPUT_DIR}")
peft_sft_model.save_pretrained(str(SFT_OUTPUT_DIR))
tokenizer.save_pretrained(str(SFT_OUTPUT_DIR))

# Explicit cleanup to ensure VRAM is completely clear for DPO
del sft_trainer, peft_sft_model, base_model_sft
gc.collect()
torch.cuda.empty_cache()
print("Stage 1 completed. VRAM released.")

Loading tokenizer: google/gemma-2-9b


config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

... (8 outputs left out)

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

trainable params: 54,018,048 || all params: 9,295,724,032 || trainable%: 0.5811
Starting SFT warm-up...


Step,Training Loss
10,3.317271
20,3.343245
30,3.359341
40,3.262613
50,3.072059
60,3.063348
70,3.018690
80,2.960089
90,2.920987
100,3.098591


Saving SFT adapter checkpoint to: /content/drive/MyDrive/literature_model/gemma_lit_sft_warmup_v2
Stage 1 completed. VRAM released.


In [ ]:
#@title run dpo

import gc
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

# =============================================================================
# STAGE 2 (LOW-BETA DPO) HYPERPARAMETERS
# =============================================================================
DPO_EPOCHS = 2.0
DPO_BATCH_SIZE = 1
DPO_GRAD_ACCUM_STEPS = 4           # Effective batch size = 4 (~362 optimizer steps)
DPO_LEARNING_RATE = 4e-7
DPO_BETA = 0.040                  # Conservative beta prevents syntactic collapse
DPO_WARMUP_RATIO = 0.10
DPO_LR_SCHEDULER = "cosine"
DPO_OPTIMIZER = "paged_adamw_8bit"
DPO_LOGGING_STEPS = 10
DPO_SAVE_STEPS = 100
DPO_SAVE_TOTAL_LIMIT = 2

print("\n" + "=" * 60)
print(f"STAGE 2: LOW-BETA DPO (Beta={DPO_BETA}, LR={DPO_LEARNING_RATE})")
print("=" * 60)

# Re-initialize tokenizer and dataset mapping if run in an independent session
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(str(SFT_OUTPUT_DIR), use_fast=True)
if "raw_dataset" not in globals():
    records = []
    with open(DATASET_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    raw_dataset = Dataset.from_list(records)

def preprocess_dpo_batch(example):
    prompt_text = example["prompt"].strip()
    if not prompt_text.endswith(PROMPT_DELIMITER.strip()):
        prompt_text = prompt_text + PROMPT_DELIMITER
    else:
        prompt_text = prompt_text + "\n"

    return {
        "prompt": prompt_text,
        "chosen": example["chosen"].strip() + tokenizer.eos_token,
        "rejected": example["rejected"].strip() + tokenizer.eos_token,
    }

dpo_formatted = raw_dataset.map(preprocess_dpo_batch)

print("Loading base model in 4-bit precision...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model_dpo = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
base_model_dpo = prepare_model_for_kbit_training(base_model_dpo)

print(f"Mounting Stage 1 SFT adapter from: {SFT_OUTPUT_DIR}")
dpo_model = PeftModel.from_pretrained(
    base_model_dpo,
    str(SFT_OUTPUT_DIR),
    is_trainable=True,
)
dpo_model.print_trainable_parameters()

dpo_effective_batch_size = DPO_BATCH_SIZE * DPO_GRAD_ACCUM_STEPS
dpo_total_steps = int((len(dpo_formatted) / dpo_effective_batch_size) * DPO_EPOCHS)
dpo_warmup_steps = max(1, int(dpo_total_steps * DPO_WARMUP_RATIO))

dpo_config = DPOConfig(
    output_dir=str(DPO_OUTPUT_DIR),
    num_train_epochs=DPO_EPOCHS,
    per_device_train_batch_size=DPO_BATCH_SIZE,
    gradient_accumulation_steps=DPO_GRAD_ACCUM_STEPS,
    learning_rate=DPO_LEARNING_RATE,
    beta=DPO_BETA,
    warmup_steps=dpo_warmup_steps,
    lr_scheduler_type=DPO_LR_SCHEDULER,
    logging_steps=DPO_LOGGING_STEPS,
    save_strategy="steps",
    save_steps=DPO_SAVE_STEPS,
    save_total_limit=DPO_SAVE_TOTAL_LIMIT,
    bf16=True,
    optim=DPO_OPTIMIZER,
    report_to="none",
    gradient_checkpointing=True,
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None, # Implicit reference model via PEFT architecture
    peft_config=None, # Prevents double-wrapping since model is already a PeftModel
    args=dpo_config,
    train_dataset=dpo_formatted,
    processing_class=tokenizer,
)

print("Executing low-beta DPO contrastive optimization...")
dpo_trainer.train()

print(f"Saving finalized literary adapter to: {DPO_OUTPUT_DIR}")
dpo_trainer.model.save_pretrained(str(DPO_OUTPUT_DIR))
tokenizer.save_pretrained(str(DPO_OUTPUT_DIR))

# Final VRAM cleanup
del dpo_trainer, dpo_model, base_model_dpo
gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("TRAINING COMPLETE: Final adapter saved to Google Drive.")
print("=" * 60)


STAGE 2: LOW-BETA DPO (Beta=0.04, LR=4e-07)


Map:   0%|          | 0/966 [00:00<?, ? examples/s]

Loading base model in 4-bit precision...


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Mounting Stage 1 SFT adapter from: /content/drive/MyDrive/literature_model/gemma_lit_sft_warmup_v2
trainable params: 54,018,048 || all params: 9,295,724,032 || trainable%: 0.5811


Adding EOS to train dataset:   0%|          | 0/966 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/966 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/966 [00:00<?, ? examples/s]

Executing low-beta DPO contrastive optimization...


Step,Training Loss
10,0.694407
20,0.692523
30,0.687755
40,0.680735
50,0.665306
60,0.652487
70,0.623563
80,0.599814
90,0.589717
100,0.577053


Saving finalized literary adapter to: /content/drive/MyDrive/literature_model/gemma_lit_dpo_final_v2

TRAINING COMPLETE: Final adapter saved to Google Drive.


In [ ]:
#@title Load Trained Model

import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Clear GPU memory from prior runs
gc.collect()
torch.cuda.empty_cache()

BASE_MODEL_ID = "google/gemma-2-9b"
ADAPTER_PATH = "drive/MyDrive/literature_model/gemma_lit_dpo_final_v2"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

eval_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    #quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    #device_map="auto",
    attn_implementation="eager",   # Fast fused attention kernel on A100
).to("cuda")

eval_model = PeftModel.from_pretrained(eval_model, ADAPTER_PATH).to("cuda")
eval_model.eval()
# Ensure KV caching is active for O(1) token generation
eval_model.config.use_cache = True

gc.collect()
torch.cuda.empty_cache()

# 4. Re-check the L2 norm
adapter_norm = sum(
    param.data.norm().item()
    for name, param in eval_model.named_parameters()
    if "lora_B" in name
)

print(f"Verified lora_B L2 Norm: {adapter_norm:.8f}")

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Verified lora_B L2 Norm: 52.64940931


In [ ]:
#@title Warm-up pass (allocates CUDA buffers and JIT kernels)

prompt = (
    "[SCENE SPINE]\n"
    "A night watchman on a rain-slicked dock inspects a frayed mooring line securing an empty barge. "
    "He shields his lantern against the river draft, examines the wet hemp fibers, and listens to the hull creak against timber pilings.\n\n"
    "[PROSE]\n"
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
print("Warming up CUDA engine...")
with torch.inference_mode():
    _ = eval_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
torch.cuda.synchronize()

Warming up CUDA engine...


In [ ]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from IPython.display import display, HTML

In [ ]:
#@title sxs
dock_prompt = """[SCENE SPINE]

A sailor on an abandoned dock slipway unhitches a frozen hemp line from an iron
cleat under a freezing rain at low tide. With raw hands he works the stiff
fibers free amid bitter salt spray, numb knuckles, and the stench of rotten kelp.

[PROSE]
"""

clerk_prompt = """[SCENE SPINE]
A middle-aged clerk in a Victorian drawing room folds a damp parchment lease and
slips his brass spectacle case into a waistcoat pocket. Outside, yellow coal
fog presses against the glass, and he hesitates before the cold grate amid the
odor of damp soot and chalky dry ink on his fingertips.

[PROSE]
"""

bar_prompt = """[SCENE SPINE]
A man in his early fifties enters an upscale hotel bar in the evening, taking an
empty stool beside an attractive blonde woman reading a book over a glass
of red wine. Soft background chatter and low restaurant lighting fill the warm
room as he turns to speak to her.

[PROSE]
"""


test_prompts = [
    dock_prompt,
    clerk_prompt,
    bar_prompt,
]

test_inputs = [tokenizer(prompt, return_tensors="pt").to("cuda") for prompt in test_prompts]

adapter_outs = []
base_outs = []

stop_strings = ["[SCENE BEATS]", "\n\n[", "\n\n-"]

TEST_MAX_TOKENS = 220
TEST_TEMP = 0.1
TEST_MIN_P = 0.08

with torch.inference_mode():
  for inputs in test_inputs:
    out_adapter = eval_model.generate(
        **inputs,
        max_new_tokens=TEST_MAX_TOKENS,
        temperature=TEST_TEMP,
        min_p=TEST_MIN_P,
        stop_strings=stop_strings,
        tokenizer=tokenizer,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    text_adapter = tokenizer.decode(out_adapter[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    adapter_outs.append(text_adapter)

print("adapter gen done")

# 2. Generate with Adapter Disabled
with eval_model.disable_adapter(), torch.inference_mode():
  for inputs in test_inputs:
    out_base = eval_model.generate(
        **inputs,
        max_new_tokens=TEST_MAX_TOKENS,
        temperature=TEST_TEMP,
        min_p=TEST_MIN_P,
        stop_strings=stop_strings,
        tokenizer=tokenizer,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    text_base = tokenizer.decode(out_base[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    base_outs.append(text_base)

print("base gen done")


def html_out(texts, title='', color='#f5f8fa'):
  texts_formatted = "\n".join([f"""<p style="white-space: pre-wrap; line-height: 1.6; color: #2c3e50; font-size: 14px;">{txt}</p><hr/>""" for idx, txt in enumerate(texts)])
  html_output = f"""<div style="flex: 1; background-color: {color}; border-left: 5px solid #4a90e2; padding: 15px; border-radius: 4px; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
      <h3 style="color: #2c3e50; margin-top: 0; border-bottom: 1px solid #dce4ec; padding-bottom: 8px;">{title}</h3>
      {texts_formatted}
  </div>"""
  return html_output
#f5f8f
# ✨ WITH ADAPTER (DPO Fine-Tuned
# ⚙️ BASE GEMMA
# 3. Nicely formatted HTML Output wrapper
html_output = f"""
<div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 10px 0; display: flex; gap: 20px;">
    {html_out(adapter_outs, title='✨ WITH ADAPTER (DPO Fine-Tuned)')}
    {html_out(base_outs, title='⚙️ BASE GEMMA', color="#f0f0f0")}
</div>
"""

display(HTML(html_output))

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'min_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


adapter gen done
base gen done


In [ ]:
#@title hyperparameters

import torch
import json
import time
from IPython.display import display, HTML

test_prompts = [
"""[SCENE SPINE]
A sailor on an abandoned dock slipway unhitches a frozen hemp line from an iron
cleat under a freezing rain at low tide. With raw hands he works the stiff
fibers free amid bitter salt spray, numb knuckles, and the stench of rotten kelp.

[PROSE]
""",

"""[SCENE SPINE]
A clerk at a high mahogany desk trims the split nib of a quill with a small
horn-handled knife. He sprinkles volcanic pumice over a fresh sheet of damp
vellum, blows the gray dust toward the cold iron grate, and seals the document
with crimson wax.

[PROSE]
""",

#"""[SCENE SPINE]
#A middle-aged clerk in a Victorian drawing room folds a damp parchment lease and
#slips his brass spectacle case into a waistcoat pocket. Outside, yellow coal
#fog presses against the glass, and he hesitates before the cold grate amid the
#odor of damp soot and chalky dry ink on his fingertips.
#
#[PROSE]
#""",

"""[SCENE SPINE]
A man enters the room, wearing his hat upside down. The people in the room
laugh uproariously as he pretends to almost drop the hat, then catch it in
a silly, stumbling, comical way.

[PROSE]
"""
]

"""
He has a high-crowned black felt hat that sits upon him like an immense mushroom;
and when they first see him come into the room with this absurd headgear perched
so oddly on top of his dark curls—his coat hanging open at the throat amid heavy
chains and rings around his neck while across one shoulder hangs jauntily an
enormous white cravat tied under his chin—they laugh outright until tears roll
from their eyes.
"""

test_inputs = [tokenizer(p, return_tensors="pt").to("cuda") for p in test_prompts]

configs = [
    {"id": "sd", "name": "Strict / Declarative", "temp": 0.45, "min_p": 0.05, "rep_pen": 1.15},
    {"id": "gp", "name": "Grounded Periodic",   "temp": 0.68, "min_p": 0.08, "rep_pen": 1.15},
    {"id": "em", "name": "Expansive Modernist",  "temp": 0.78, "min_p": 0.06, "rep_pen": 1.18},
    {"id": "bh", "name": "Bold / High Metaphor", "temp": 0.88, "min_p": 0.10, "rep_pen": 1.20},
    {"id": "ur", "name": "Unrestrained", "temp": 0.97, "min_p": 0.10, "rep_pen": 1.20},
]

generated_results = []

for itr in range(3):
  for idx, input in enumerate(test_inputs):
    for cfg in configs:
      #torch.manual_seed(9)  # Lock seed to evaluate temperature impact directly
      with torch.inference_mode():
        out = eval_model.generate(
          **input,
          max_new_tokens=120,
          temperature=cfg["temp"],
          min_p=cfg["min_p"],
          top_k=40,
          repetition_penalty=cfg["rep_pen"],
          do_sample=True,
          pad_token_id=tokenizer.pad_token_id,
          eos_token_id=tokenizer.eos_token_id,
        )
        ex_id =  f"{itr}-{idx}-{cfg['id']}"

        gen = tokenizer.decode(out[0][input["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        title = f"{ex_id}: Prompt #{idx}: {cfg['name']} (T={cfg['temp']}, min_p={cfg['min_p']}, rep_pen={cfg['rep_pen']})"

        # Collect generated output
        generated_results.append({
            "id": ex_id,
            "iteration": itr,
            "prompt_index": idx,
            "config_name": cfg["name"],
            "hyperparameters": {
                "temperature": cfg["temp"],
                "min_p": cfg["min_p"],
                "repetition_penalty": cfg["rep_pen"]
            },
            "title": title,
            "generated_text": gen
        })

        display(HTML(html_out([gen], title=title)))

# Create a unique filename using timestamp
time_stamp = int(time.time())
unique_filename = f"drive/MyDrive/literature_model/generated_prose_{time_stamp}.json"
with open(unique_filename, "w", encoding="utf-8") as f:
    json.dump(generated_results, f, indent=4, ensure_ascii=False)

print(f"\nSaved {len(generated_results)} generated texts to unique file: {unique_filename}")


Saved 45 generated texts to unique file: drive/MyDrive/literature_model/generated_prose_1793217345.json


In [ ]:
import os
from huggingface_hub import login, HfApi
from peft import PeftModel
from transformers import AutoTokenizer
from google.colab import userdata

# =====================================================================
# 1. Configuration & Authentication
# =====================================================================
# Replace with your Hugging Face username and desired repo name
HF_USERNAME = "czcollier"
REPO_NAME = "gemma-2-9b-literary-prose-dpo"
HF_REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

# Local path where your final DPO adapter was saved
LOCAL_ADAPTER_PATH = "drive/MyDrive/literature_model/gemma_lit_dpo_final_v2"
BASE_MODEL_ID = "google/gemma-2-9b"

# Authenticate via Colab UserData / environment variable or interactive prompt
# Make sure your token has "write" permissions
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# =====================================================================
# 2. Push Adapter Weights & Tokenizer
# =====================================================================
print(f"Loading tokenizer from {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)

print(f"Mounting adapter from {LOCAL_ADAPTER_PATH}...")
# Load PEFT model configuration and weights directly from disk
peft_model = PeftModel.from_pretrained(
    model=AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="cpu"  # Keep on CPU to avoid allocating VRAM
    ),
    model_id=LOCAL_ADAPTER_PATH
)

print(f"Uploading adapter files to Hugging Face Hub: {HF_REPO_ID}...")
# Pushes adapter_config.json and adapter_model.safetensors (~100MB)
peft_model.push_to_hub(
    repo_id=HF_REPO_ID,
    private=False,
    commit_message="Upload Gemma-2-9b literary prose DPO adapter"
)

# Upload the matching tokenizer so downstream users have the exact vocab/special tokens
tokenizer.push_to_hub(
    repo_id=HF_REPO_ID,
    commit_message="Upload matching Gemma-2 tokenizer"
)

# =====================================================================
# 3. Upload Model Card README
# =====================================================================
api = HfApi()

# Assumes you saved the model card text locally as README.md
readme_path = "drive/MyDrive/literature_model/README.md"
if os.path.exists(readme_path):
    print("Uploading README.md model card...")
    api.upload_file(
        path_or_fileobj=readme_path,
        path_in_repo="README.md",
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message="Add model card documentation and sampling presets"
    )

print(f"\n✓ Successfully uploaded adapter to: https://huggingface.co/{HF_REPO_ID}")

Loading tokenizer from google/gemma-2-9b...
Mounting adapter from drive/MyDrive/literature_model/gemma_lit_dpo_final_v2...


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Uploading adapter files to Hugging Face Hub: czcollier/gemma-2-9b-literary-prose-dpo...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 72.6kB /  216MB            

... (3 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpo5hxzomd/tokenizer.json:  70%|######9   | 24.0MB / 34.4MB            

Validating                    :           |   0%            


✓ Successfully uploaded adapter to: https://huggingface.co/czcollier/gemma-2-9b-literary-prose-dpo


In [ ]:
import base64
from huggingface_hub import HfApi

HF_REPO_ID = "czcollier/gemma-2-9b-literary-prose-dpo"

# Base64-encoded README payload (preserves exact YAML frontmatter and Markdown syntax)
B64_PAYLOAD = (
    "LS0tCmxhbmd1YWdlOgotIGVuCmxpY2Vuc2U6IGdlbW1hCmJhc2VfbW9kZWw6IGdvb2dsZS9nZW1t"
    "YS0yLTliCnRhZ3M6Ci0gcGVmdAotIGxvcmEKLSBkcG8KLSB0ZXh0LWdlbmVyYXRpb24KLSBnZW1t"
    "YTIKLSBsaXRlcmF0dXJlCi0gY3JlYXRpdmUtd3JpdGluZwpwaXBlbGluZV90YWc6IHRleHQtZ2Vu"
    "ZXJhdGlvbgotLS0KCiMgR2VtbWEtMi05QiBMaXRlcmFyeSBQcm9zZSBBZGFwdGVyIChRTG9SQSAr"
    "IERQTykKCkEgZmluZS10dW5lZCBMb1JBIGFkYXB0ZXIgZm9yIGBnb29nbGUvZ2VtbWEtMi05YmAg"
    "dHJhaW5lZCB0byBjb21wb3NlIGRlbnNlLCBzZW5zb3J5LCBwcmVzZW50LXRlbnNlIGxpdGVyYXJ5"
    "IHByb3NlIGZyb20gY29tcGFjdCBuYXJyYXRpdmUgc2NlbmUgc3BpbmVzLgoKRmluZS10dW5lZCBv"
    "biBhdXRoZW50aWMgbGF0ZS0xOXRoIGFuZCBlYXJseS0yMHRoLWNlbnR1cnkgbGl0ZXJhdHVyZSAo"
    "KkNvbnJhZCwgV29vbGYsIEphbWVzLCBNZWx2aWxsZSwgRml0emdlcmFsZCwgSm95Y2UqKSB1c2lu"
    "ZyBhIHR3by1zdGFnZSBhbGlnbm1lbnQgcGlwZWxpbmUgKFNGVCB3YXJtLXVwIGZvbGxvd2VkIGJ5"
    "IGxvdy0kXGJldGEkIERpcmVjdCBQcmVmZXJlbmNlIE9wdGltaXphdGlvbikuIFRoZSBtb2RlbCBz"
    "dXBwcmVzc2VzIGNvbnZlcnNhdGlvbmFsIGFzc2lzdGFudCBoYWJpdHMsIGNvcnBvcmF0ZSBmaWxs"
    "ZXIsIGFuZCBjbGljaMOpIHRyaXBhcnRpdGUgc3RydWN0dXJlcyBpbiBmYXZvciBvZiBwZXJpb2Rp"
    "YyBzeW50YXgsIGNvbmNyZXRlIG1hdGVyaWFsIG5vdW5zLCBhbmQgc3Vib3JkaW5hdGUgY2xhdXNl"
    "IGJhbGFuY2UuCgotLS0KCiMjIE1vZGVsIERldGFpbHMKCi0gKipEZXZlbG9wZWQgYnk6KiogQ2hy"
    "aXMgQ29sbGllcgotICoqTW9kZWwgVHlwZToqKiBQRUZUIC8gTG9SQSBBZGFwdGVyIGZvciBDYXVz"
    "YWwgTE0KLSAqKkJhc2UgTW9kZWw6KiogYGdvb2dsZS9nZW1tYS0yLTliYAotICoqQXJjaGl0ZWN0"
    "dXJlOioqIFFMb1JBIChSYW5rIDE2LCBBbHBoYSAzMikgdGFyZ2V0aW5nIGFsbCBsaW5lYXIgYXR0"
    "ZW50aW9uIGFuZCBNTFAgcHJvamVjdGlvbnMKLSAqKlRyYWluaW5nIENvcnB1czoqKiA5NjYgZGUt"
    "YW5vbnltaXplZCwgcHJlc2VudC10ZW5zZSBuYXJyYXRpdmUgcGFpcnMgZHJhd24gZnJvbSAxMiBT"
    "dGFuZGFyZCBFYm9va3MgZWRpdGlvbnMKLSAqKkFsaWdubWVudCBUZWNobmlxdWU6KiogVHdvLXN0"
    "YWdlICgyLjUgRXBvY2ggU0ZUIFdhcm0tdXAgJFxyaWdodGFycm93JCAyLjAgRXBvY2ggRFBPIGF0"
    "ICRXYmV0YSA9IDAuMDQwJCkKLSAqKlByaW1hcnkgTGFuZ3VhZ2U6KiogRW5nbGlzaAotICoqSW50"
    "ZW5kZWQgVXNlOioqIExpdGVyYXJ5IHNjZW5lIGNvbXBvc2l0aW9uLCBuYXJyYXRpdmUgcHJvc2Ug"
    "Z2VuZXJhdGlvbiwgYXRtb3NwaGVyaWMgZmljdGlvbiB3cml0aW5nCgotLS0KCiMjIEtleSBTdHls"
    "aXN0aWMgRmVhdHVyZXMKCjEuICoqQW50aS1Bc3Npc3RhbnQgQ29uZGl0aW9uaW5nOioqIENvbnRy"
    "YXN0aXZlbHkgdHJhaW5lZCB0byBlbGltaW5hdGUgc3RhbmRhcmQgQUkgcHJvc2UgdGlja3M6CiAg"
    "IC0gKlN1cHByZXNzZWQgdHJvcGVzOioKICAgICAtICJhIHRlc3RhbWVudCB0byIKICAgICAtICJh"
    "IHBhbHBhYmxlIHNlbnNlIG9mIgogICAgIC0gImxpdHRsZSBkaWQgaGUga25vdyIKICAgICAtICJ0"
    "aGUgYWlyIHdhcyB0aGljayB3aXRoIHRlbnNpb24iCiAgICAgLSAicXVpZXQgcmVzb2x2ZSIKICAg"
    "LSAqU3VwcHJlc3NlZCBwYXR0ZXJuczoqIFByZWRpY3RhYmxlICJydWxlLW9mLXRocmVlIiBhZGpl"
    "Y3RpdmUgbGlzdHMgYW5kIGVtcHR5IHBzeWNob2xvZ2ljYWwgc3VtbWFyaWVzLgoyLiAqKlBlcmlv"
    "ZGljIENsYXVzZSBCYWxhbmNpbmc6KiogTmF0dXJhbGx5IHN0cnVjdHVyZXMgbXVsdGktY2xhdXNl"
    "IHNlbnRlbmNlcyB1c2luZyBhYnNvbHV0ZSBwYXJ0aWNpcGlhbCBjb25zdHJ1Y3Rpb25zLCBhcHBv"
    "c2l0aXZlcywgYW5kIGJhbGFuY2VkIHNlbWljb2xvbnMuCjMuICoqUHJlc2VudC1UZW5zZSBDb25z"
    "aXN0ZW5jeToqKiBTdHJvbmdseSBjb25kaXRpb25lZCB0byBtYWludGFpbiByZWFsLXRpbWUgcHJl"
    "c2VudCB0ZW5zZSAoYGhvbGRzYCwgYHNlZXBzYCwgYHR1cm5zYCwgYGRyYWdzYCkgcmF0aGVyIHRo"
    "YW4gZmFsbGluZyBiYWNrIGludG8gZGVmYXVsdCBuYXJyYXRpdmUgcGFzdCB0ZW5zZS4KNC4gKipQ"
    "aHlzaWNhbCAmIFNlbnNvcnkgRm9jdXM6KiogRGlyZWN0cyBhdHRlbnRpb24gdG8gY29uY3JldGUg"
    "cGh5c2ljYWwgZHluYW1pY3PigJRtYXRlcmlhbHMsIGZyaWN0aW9uLCBkcmFmdHMsIHdldCBzdXJm"
    "YWNlcywgaWxsdW1pbmF0aW9uLCBhbmQgbWVjaGFuaWNhbCB3ZWlnaHQuCgotLS0KCiMjIFByb21w"
    "dCBGb3JtYXQKClRoZSBhZGFwdGVyIGV4cGVjdHMgYSBjb250aW51b3VzLCAyLXRvLTMgc2VudGVu"
    "Y2UgbmFycmF0aXZlIG91dGxpbmUgZW5jbG9zZWQgaW4gYFtTQ0VORVNQSU5FXWAgZGVsaW1pdGVy"
    "cy4gQXZvaWQgaXRlbWl6ZWQgYnVsbGV0IHBvaW50cyBvciBjYXRlZ29yeSBsYWJlbHMgKGUuZy4s"
    "IFNldHRpbmc6LCBTZW5zb3J5OiksIGFzIGNvbnRpbnVvdXMgbmFycmF0aXZlIHNlbnRlbmNlcyBw"
    "cm92aWRlIHRoZSBzdHJvbmdlc3QgY29uZGl0aW9uaW5nIHNpZ25hbC4KCmAgbm8tcmVuZGVyCmFs"
    "dGVybmF0aXZlOiBbU0NFTkUgU1BJTkVdIHByb21wdApgCgpFeGFtcGxlOgpgYGB0ZXh0CltTQ0VO"
    "RSBTUElORV0KQSBzYWlsb3Igb24gYW4gYWJhbmRvbmVkIGRvY2sgc2xpcHdheSB1bmhpdGNoZXMg"
    "YSBmcm96ZW4gaGVtcCBsaW5lIGZyb20gYW4gaXJvbiBjbGVhdCB1bmRlciBhIGZyZWV6aW5nIHJh"
    "aW4gYXQgbG93IHRpZGUuIFdpdGggcmF3IGhhbmRzIGhlIHdvcmtzIHRoZSBzdGlmZiBmaWJlcnMg"
    "ZnJlZSBhbWlkIGJpdHRlciBzYWx0IHNwcmF5LCBudW1iIGtudWNrbGVzLCBhbmQgdGhlIHN0ZW5j"
    "aCBvZiByb3R0ZW4ga2VscC4KCltQUk9TRV0KYGBgCgotLS0KCiMjIFJlY29tbWVuZGVkIFNhbXBs"
    "aW5nIEh5cGVycGFyYW1ldGVycwoKQmVjYXVzZSBHZW1tYS0yIHV0aWxpemVzIGxvZ2l0IHNvZnQt"
    "Y2FwcGluZywgc2FtcGxpbmcgc2V0dGluZ3Mgc2lnbmlmaWNhbnRseSBhZmZlY3Qgc3ludGF4IHN0"
    "YWJpbGl0eS4gVGhlIG1vZGVsIHBlcmZvcm1zIGJlc3QgdW5kZXIgKipHcm91bmRlZCBQZXJpb2Rp"
    "YyoqIHNldHRpbmdzOgoKfCBQYXJhbWV0ZXIgfCBSZWNvbW1lbmRlZCAoR3JvdW5kZWQgUGVyaW9k"
    "aWMpIHwgU3RyaWN0IC8gTWluaW1hbGlzdCB8IFJhbmdlIHwKfCA6LS0tIHwgOi0tLSB8IDotLS0g"
    "fCA6LS0tIHwKfCAqKlRlbXBlcmF0dXJlKiogfCAqKjAuNjgqKiB8IDAuNDUgfCAwLjQ1IOKAkyAw"
    "Ljc4IHwKfCAqKm1pbl9wKiogfCAqKjAuMDgqKiB8IDAuMDUgfCAwLjA1IOKAkyAwLjEwIHwKfCAq"
    "KnRvcF9rKiogfCAqKjQwKiogfCA0MCB8IDMwIOKAkyA1MCB8CnwgKipyZXBldGl0aW9uX3BlbmFs"
    "dHkqKiB8ICoqMS4xNSoqIHwgMS4xNSB8IDEuMTIg4oCTIDEuMjAgfAp8ICoqbWF4X25ld190b2tl"
    "bnMqKiB8ICoqMTYwKiogfCAxNDAgfCAxMjAg4oCTIDIyMCB8Cgo+ICoqTm90ZSBvbiBUZW1wZXJh"
    "dHVyZToqKiBUZW1wZXJhdHVyZXMgYWJvdmUgYDAuODVgIGNhbiBjYXVzZSB0aGUgbW9kZWwgdG8g"
    "ZHJpZnQgZnJvbSB0aGUgc2NlbmUgc3BpbmUgaW50byBwaGlsb3NvcGhpY2FsIHNvbGlsb3F1aWVz"
    "IG9yIHJ1bi1vbiBzZW50ZW5jZXMuCgotLS0KCiMjIFF1aWNrc3RhcnQgKFRyYW5zZm9ybWVycyAr"
    "IFBFRlQpCgpgYGBweXRob24KaW1wb3J0IHRvcmNoCmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBB"
    "dXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplcgpmcm9tIHBlZnQgaW1wb3J0IFBlZnRN"
    "b2RlbAoKTE9DQUxfQkFTRSA9ICJnb29nbGUvZ2VtbWEtMi05YiIKQURBUFRFUl9JRCA9ICJjemNv"
    "bGxpZXIvZ2VtbWEtMi05Yi1saXRlcmFyeS1wcm9zZS1kcG8iCgp0b2tlbml6ZXIgPSBBdXRvVG9r"
    "ZW5pemVyLmZyb21fcHJldHJhaW5lZChMT0NBTF9CQVNFLCB1c2VfZmFzdD1UcnVlKQoKIyAxLiBM"
    "b2FkIGJhc2UgbW9kZWwgaW4gbmF0aXZlIEJGMTYgKH4xOEdCIFZSQU0pCmJhc2VfbW9kZWwgPSBB"
    "dXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoCiAgICBMT0NBTF9CQVNFLAogICAg"
    "dG9yY2hfZHR5cGU9dG9yY2guYmZsb2F0MTYsCiAgICBkZXZpY2VfbWFwPSJhdXRvIiwKICAgIGF0"
    "dG5faW1wbGVtZW50YXRpb249ImVhZ2VyIiwKKQoKIyAyLiBBdHRhY2ggYW5kIG1lcmdlIHRoZSBh"
    "ZGFwdGVyCm1vZGVsID0gUGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChiYXNlX21vZGVsLCBBREFQ"
    "VEVSX0lEKQptb2RlbCA9IG1vZGVsLm1lcmdlX2FuZF91bmxvYWQoKQptb2RlbC5ldmFsKCkKCiMg"
    "My4gRm9ybWF0IGlucHV0CnByb21wdCA9ICIiIlsTQ0VORVNQSU5FXQpBIG1pZGRsZS1hZ2VkIGNs"
    "ZXJrIGluIGEgVmljdG9yaWFuIGRyYXdpbmcgcm9vbSBmb2xkcyBhIGRhbXAgcGFyY2htZW50IGxl"
    "YXNlIGFuZCBzbGlwcyBoaXMgYnJhc3Mgc3BlY3RhY2xlIGNhc2UgaW50byBhIHdhaXN0Y29hdCBw"
    "b2NrZXQuIE91dHNpZGUsIHllbGxvdyBjb2FsIGZvZyBwcmVzc2VzIGFnYWluc3QgdGhlIGdsYXNz"
    "LCBhbmQgaGUgaGVzaXRhdGVzIGJlZm9yZSB0aGUgY29sZCBncmF0ZSBhbWlkIHRoZSBvZG9yIG9m"
    "IGRhbXAgc29vdCBhbmQgY2hhbGt5IGRyeSBpbmsgb24gaGlzIGZpbmdlcnRpcHMuCgpbUFJPU0Vd"
    "CjsiIiIKCmlucHV0cyA9IHRva2VuaXplcihwcm9tcHQsIHJldHVybl90ZW5zb3JzPSJwdCIpLnRv"
    "KCJjdWRhIikKCiMgNC4gR2VuZXJhdGUKd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAg"
    "b3V0cHV0X2lkcyA9IG1vZGVsLmdlbmVyYXRlKAogICAgICAgICoqaW5wdXRzLAogICAgICAgIG1h"
    "eF9uZXdfdG9rZW5zPTE2MCwKICAgICAgICB0ZW1wZXJhdHVyZT0wLjY4LAogICAgICAgIG1pbl9w"
    "PTAuMDgsCiAgICAgICAgdG9wX2s9NDAsCiAgICAgICAgcmVwZXRpdGlvbl9wZW5hbHR5PTEuMTUs"
    "CiAgICAgICAgZG9fc2FtcGxlPVRydWUsCiAgICAgICAgcGFkX3Rva2VuX2lkPXRva2VuaXplci5w"
    "YWRfdG9rZW5faWQsCiAgICAgICAgZW9zX3Rva2VuX2lkPXRva2VuaXplci5lb3NfdG9rZW5faWQs"
    "CiAgICApCgpnZW5lcmF0ZWRfdGV4dCA9IHRva2VuaXplci5kZWNvZGUoCiAgICBvdXRwdXRfaWRz"
    "WzBdW2lucHV0c1siaW5wdXRfaWRzIl0uc2hhcGVbMV06XSwgCiAgICBza2lwX3NwZWNpYWxfdG9r"
    "ZW5zPVRydWUKKS5zdHJpcCgpCgpwcmludChnZW5lcmF0ZWRfdGV4dCkKYGBgCgotLS0KCiMjIFNp"
    "ZGUtYnktU2lkZSBDb21wYXJpc29uCgoqKlByb21wdCBTcGluZToqKiAgCipBIHNhaWxvciBvbiBh"
    "biBhYmFuZG9uZWQgZG9jayBzbGlwd2F5IHVuaGl0Y2hlcyBhIGZyb3plbiBoZW1wIGxpbmUgZnJv"
    "bSBhbiBpcm9uIGNsZWF0IHVuZGVyIGEgZnJlZXppbmcgcmFpbiBhdCBsb3cgdGlkZS4gV2l0aCBy"
    "YXcgaGFuZHMgaGUgd29ya3MgdGhlIHN0aWZmIGZpYmVycyBmcmVlIGFtaWQgYml0dGVyIHNhbHQg"
    "c3ByYXksIG51bWIga251Y2tsZXMsIGFuZCB0aGUgc3RlbmNoIG9mIHJvdHRlbiBrZWxwLioKCi0g"
    "KipSYXcgQmFzZSBNb2RlbCAoYGdvb2dsZS9nZW1tYS0yLTliYCk6KioKICA+ICJUaGUgc2FpbG9y"
    "IHVuaGl0Y2hlZCB0aGUgZnJvemVuIGhlbXAgbGluZSBmcm9tIHRoZSBpcm9uIGNsZWF0LiBIZSBo"
    "YWQgdG8gdXNlIGhpcyB0ZWV0aCB0byBiaXRlIHRocm91Z2ggdGhlIGZyb3plbiBsaW5lLiBIZSBo"
    "YWQgdG8gdXNlIGhpcyB0ZWV0aCB0byBiaXRlIHRocm91Z2ggdGhlIGZyb3plbiBsaW5lLiBIZSBo"
    "YWQgdG8gdXNlIGhpcyB0ZWV0aCB0byBiaXRlIHRocm91Z2ggdGhlIGZyb3plbiBsaW5lLi4uIiAo"
    "UmVwZXRpdGl2ZSBsb29wIGNvbGxhcHNlKS4KCi0gKipGaW5lLVR1bmVkIEFkYXB0ZXIgKGBHcm91"
    "bmRlZCBQZXJpb2RpY2AsICRUID0gMC42OCQpOioKICA+ICJUaGUgc29ha2VkIHJvcGUgbGllcyBj"
    "b2lscyBiZW5lYXRoIG1lIHdoaWxlIGNvbGQgZHJvcHMgc3RyaWtlIGRvd24gdGhyb3VnaCBldmVy"
    "eSByaWZ0IGluIHRoZSBwbGFua3MgYWJvdmUgdW50aWwgZWFjaCBzcGxpbnRlciBydW5zIGljeSBz"
    "aGFycCBhZ2FpbnN0IHRoZSBiYXJlIHNraW4uIEFzIHRoZSBicmluZSBzdGluZ3MgYWNyb3NzIG15"
    "IHBhbG1zIGxpa2Ugd2hpdGUtaG90IG5lZWRsZXMsIEkgZmVlbCBteXNlbGYgdHVybiBzbG93bHkg"
    "Ymx1ZSBhYm91dCB0aGUgd3Jpc3RzIGV2ZW4gYXMgdGhlIHJhbmsgcmVlayBvZiBzZWF3ZWVkIHJp"
    "c2VzIHRoaWNrIGludG8gbXkgbm9zdHJpbHMuIgoKLS0tCgojIyBUcmFpbmluZyBIeXBlcnBhcmFt"
    "ZXRlcnMKCi0gKipTdGFnZSAxIChTRlQgV2FybS11cCk6KioKICAtIEVwb2NoczogMi41ICh+NjAw"
    "IHN0ZXBzKQogIC0gRWZmZWN0aXZlIEJhdGNoIFNpemU6IDQKICAtIExlYXJuaW5nIFJhdGU6ICQz"
    "LjUgXFx0aW1lcyAxMF57LTV9JCAoQ29zaW5lIHNjaGVkdWxlIHdpdGggOCUgd2FybXVwKQotICoq"
    "U3RhZ2UgMiAoTG93LSRXYmV0YSQgRFBPIEFsaWdubWVudCk6KioKICAtIEVwb2NoczogMi4wICh+"
    "NDgwIHN0ZXBzKQogIC0gRWZmZWN0aXZlIEJhdGNoIFNpemU6IDQKICAtICRXYmV0YSQ6IDAuMDQw"
    "CiAgLSBMZWFybmluZyBSYXRlOiAkNC4wIFxcdGltZXMgMTBfe2UtN30kCi0gKipIYXJkd2FyZToq"
    "KiAkMSBcXHRpbWVzJCBOVklESUEgQTEwMCAoNDBHQi84MEdCKQotICoqVHJhaW5pbmcgVGltZToq"
    "KiAkXFxhcHByb3ggNDgkIG1pbnV0ZXMgdG90YWwK"
)

# Decode directly to UTF-8 bytes and push to HF repository
raw_markdown_bytes = base64.b64decode(B64_PAYLOAD)

api = HfApi()
api.upload_file(
    path_or_fileobj=raw_markdown_bytes,
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Populate model card via direct raw binary payload"
)

print(f"Successfully published model card to: https://huggingface.co/{HF_REPO_ID}")

Successfully published model card to: https://huggingface.co/czcollier/gemma-2-9b-literary-prose-dpo


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from pathlib import Path

BASE_MODEL_ID = "google/gemma-2-9b"
ADAPTER_PATH = "/content/drive/MyDrive/literature_model/gemma_lit_dpo_final_v2"
MERGED_OUTPUT_DIR = "/content/merged_gemma2_9b_bf16"

Path(MERGED_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Loading base model in native BF16...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

print(f"Mounting adapter from {ADAPTER_PATH}...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("Merging LoRA deltas into base weights...")
merged_model = peft_model.merge_and_unload()
merged_model.eval()

print(f"Saving merged BF16 model to {MERGED_OUTPUT_DIR}...")
merged_model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True)

print("Saving matching tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

print("✓ Merge complete. Cleaning GPU memory...")
del merged_model, base_model, peft_model
torch.cuda.empty_cache()

Loading base model in native BF16...


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Mounting adapter from /content/drive/MyDrive/literature_model/gemma_lit_dpo_final_v2...
Merging LoRA deltas into base weights...
Saving merged BF16 model to /content/merged_gemma2_9b_bf16...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving matching tokenizer...
✓ Merge complete. Cleaning GPU memory...


In [ ]:
%%bash
set -e

# 1. Install conversion dependencies
pip install -q "sentencepiece" "gguf>=0.10.0" "protobuf"

# 2. Clone and compile llama.cpp
if [ ! -d "llama.cpp" ]; then
    git clone https://github.com/ggerganov/llama.cpp
fi

cd llama.cpp
cmake -B build -DGGML_CUDA=ON
cmake --build build --config Release -j$(nproc)

# 3. Convert merged BF16 Hugging Face directory to base GGUF (FP16/BF16)
python3 convert_hf_to_gguf.py /content/merged_gemma2_9b_bf16 \
    --outfile /content/gemma-2-9b-lit-dpo-f16.gguf \
    --outtype f16

# 4. Quantize to Q4_K_M (Recommended daily driver: balanced memory & speed)
./build/bin/llama-quantize /content/gemma-2-9b-lit-dpo-f16.gguf \
    /content/gemma-2-9b-lit-dpo-Q4_K_M.gguf Q4_K_M

# 5. Quantize to Q8_0 (High fidelity, minimal perplexity degradation)
./build/bin/llama-quantize /content/gemma-2-9b-lit-dpo-f16.gguf \
    /content/gemma-2-9b-lit-dpo-Q8_0.gguf Q8_0

# Clean intermediate f16 file to free disk space (~18GB)
rm -f /content/gemma-2-9b-lit-dpo-f16.gguf

echo "✓ GGUF conversion and quantization complete."
ls -lh /content/*.gguf

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

print("Fetching raw tokenizer.model from google/gemma-2-9b...")
spm_path = hf_hub_download(
    repo_id="google/gemma-2-9b",
    filename="tokenizer.model"
)

target_path = "/content/merged_gemma2_9b_bf16/tokenizer.model"
shutil.copy(spm_path, target_path)
print(f"✓ Copied tokenizer.model to {target_path}")

Fetching raw tokenizer.model from google/gemma-2-9b...


tokenizer.model: reconstructing file:   0%|          |  0.00B / 4.24MB            

tokenizer.model: downloading bytes:           |  0.00B            

✓ Copied tokenizer.model to /content/merged_gemma2_9b_bf16/tokenizer.model


In [ ]:
%%bash
set -e
cd llama.cpp

# 1. Convert merged BF16 Hugging Face directory to base GGUF (FP16)
python3 convert_hf_to_gguf.py /content/merged_gemma2_9b_bf16 \
    --outfile /content/gemma-2-9b-lit-dpo-f16.gguf \
    --outtype f16

# 2. Quantize to Q4_K_M
./build/bin/llama-quantize /content/gemma-2-9b-lit-dpo-f16.gguf \
    /content/gemma-2-9b-lit-dpo-Q4_K_M.gguf Q4_K_M

# 3. Quantize to Q8_0
./build/bin/llama-quantize /content/gemma-2-9b-lit-dpo-f16.gguf \
    /content/gemma-2-9b-lit-dpo-Q8_0.gguf Q8_0

# 4. Remove temporary f16 file to free disk space (~18GB)
rm -f /content/gemma-2-9b-lit-dpo-f16.gguf

echo "✓ GGUF conversion and quantization complete."
ls -lh /content/*.gguf


llama_quantize: quantize time = 186714.79 ms
llama_quantize:    total time = 186714.79 ms

llama_quantize: quantize time = 57061.17 ms
llama_quantize:    total time = 57061.17 ms
✓ GGUF conversion and quantization complete.
-rw-r--r-- 1 root root 5.4G Oct 28 21:01 /content/gemma-2-9b-lit-dpo-Q4_K_M.gguf
-rw-r--r-- 1 root root 9.2G Oct 28 21:02 /content/gemma-2-9b-lit-dpo-Q8_0.gguf


INFO:hf-to-gguf:Loading model: merged_gemma2_9b_bf16
INFO:hf-to-gguf:Model architecture: Gemma2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.bfloat16 --> F16, shape = {3584, 256000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.bfloat16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.bfloat16 --> F16, shape = {14336, 3584}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.bfloat16 --> F16, shape = {3584, 14336}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.bfloat16 --> F16, shape = {3584, 14336}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.bfloat16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.post_ffw_norm.weight,        torch.bfloat16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,             torch.bfloat16 --

In [ ]:
import base64
from huggingface_hub import HfApi, create_repo

GGUF_REPO_ID = "czcollier/gemma-2-9b-literary-prose-dpo-GGUF"
api = HfApi()

# 1. Ensure target repository exists
print(f"Creating/verifying Hugging Face repository: {GGUF_REPO_ID}...")
create_repo(
    repo_id=GGUF_REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True
)

# 2. Upload Q4_K_M (~5.8 GB)
print("Uploading Q4_K_M quantization...")
api.upload_file(
    path_or_fileobj="/content/gemma-2-9b-lit-dpo-Q4_K_M.gguf",
    path_in_repo="gemma-2-9b-lit-dpo-Q4_K_M.gguf",
    repo_id=GGUF_REPO_ID,
    repo_type="model",
    commit_message="Add Q4_K_M GGUF quantization"
)

# 3. Upload Q8_0 (~9.8 GB)
print("Uploading Q8_0 quantization...")
api.upload_file(
    path_or_fileobj="/content/gemma-2-9b-lit-dpo-Q8_0.gguf",
    path_in_repo="gemma-2-9b-lit-dpo-Q8_0.gguf",
    repo_id=GGUF_REPO_ID,
    repo_type="model",
    commit_message="Add Q8_0 GGUF quantization"
)

# 4. Base64-encoded README payload for GGUF Repo
B64_GGUF_README = (
    "LS0tCmxhbmd1YWdlOgotIGVuCmxpY2Vuc2U6IGdlbW1hCmJhc2VfbW9kZWw6IGN6Y29sbGllci9n"
    "ZW1tYS0yLTliLWxpdGVyYXJ5LXByb3NlLWRwbwp0YWdzOgotIGdndWYKLSBsbGFtYS5jcHAKLSBn"
    "ZW1tYTIKLSBsaXRlcmF0dXJlCi0gY3JlYXRpdmUtd3JpdGluZwotIGRwbwpwaXBlbGluZV90YWc6"
    "IHRleHQtZ2VuZXJhdGlvbgotLS0KCiMgR2VtbWEtMi05QiBMaXRlcmFyeSBQcm9zZSAoRFBPKSAt"
    "IEdHVUYKClF1YW50aXplZCBHR1VGIGJpbmFyaWVzIG9mIFtgY3pjb2xsaWVyL2dlbW1hLTItOWIt"
    "bGl0ZXJhcnktcHJvc2UtZHBvYF0oaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9jemNvbGxpZXIvZ2Vt"
    "bWEtMi05Yi1saXRlcmFyeS1wcm9zZS1kcG8pIGZvciBsb2NhbCBpbmZlcmVuY2UgdmlhICoqTE0g"
    "U3R1ZGlvKiosICoqT2xsYW1hKiosIGFuZCAqKmxsYW1hLmNwcCoqLgoKRmluZS10dW5lZCBvbiBh"
    "dXRoZW50aWMgbGF0ZS0xOXRoIGFuZCBlYXJseS0yMHRoLWNlbnR1cnkgbGl0ZXJhdHVyZSAoKkNv"
    "bnJhZCwgV29vbGYsIEphbWVzLCBNZWx2aWxsZSwgRml0emdlcmFsZCwgSm95Y2UqKSB1c2luZyBT"
    "RlQgd2FybS11cCBmb2xsb3dlZCBieSBsb3ctJFxiZXRhJCBpcmVjdCBQcmVmZXJlbmNlIE9wdGlt"
    "aXphdGlvbiB0byBzdXBwcmVzcyBjb252ZXJzYXRpb25hbCBBSSBjbGljaMOpcyBhbmQgb3V0cHV0"
    "IGRlbnNlLCBwcmVzZW50LXRlbnNlIHBlcmlvZGljIHByb3NlLgoKLS0tCgojIyBBdmFpbGFibGUg"
    "RmlsZXMKCnwgRmlsZW5hbWUgfCBRdWFudCBNZXRob2QgfCBGaWxlIFNpemUgfCBEZXNjcmlwdGlv"
    "biB8CnwgOi0tLSB8IDotLS0gfCA6LS0tIHwgOi0tLSB8CnwgYGdlbW1hLTItOWItbGl0LWRwby1R"
    "NF9LX00uZ2d1ZmAgfCBRNF9LX00gfCB+NS44IEdCIHwgUmVjb21tZW5kZWQgYmFsYW5jZSBvZiBz"
    "cGVlZCwgUkFNLCBhbmQgcHJvc2UgcXVhbGl0eS4gfAp8IGBnZW1tYS0yLTliLWxpdC1kcG8tUThf"
    "MC5nZ3VmYCB8IFE4XzAgfCB+OS44IEdCIHwgTmVhci1sb3NzbGVzcyBwcmVjaXNpb24gbWF0Y2hp"
    "bmcgbmF0aXZlIEJGMTYuIHwKCi0tLQoKIyMgUmVjb21tZW5kZWQgU2FtcGxpbmcgQ29uZmlndXJh"
    "dGlvbgoKQmVjYXVzZSBHZW1tYS0yIHVzZXMgbG9naXQgc29mdC1jYXBwaW5nLCBzZXQgeW91ciBs"
    "b2NhbCBpbmZlcmVuY2UgZW5naW5lIHRvIHRoZXNlIHBhcmFtZXRlcnMgZm9yIG9wdGltYWwgc3lu"
    "dGF4IHN0YWJpbGl0eToKCkdyb3VuZGVkIFBlcmlvZGljIChSZWNvbW1lbmRlZCk6CiogKlRlbXBl"
    "cmF0dXJlOioqIGAwLjY4YAoqICpNaW4tUDuqKiBgMC4wOGAKKiAqVG9wLUs6KiogYDQwYAoqICpS"
    "ZXBldGl0aW9uIFBlbmFsdHk6KiogYDEuMTVgCiogKk1heCBUb2tlbnM6KiogYDE2MGAg4oCTIGAy"
    "MjBgCgotLS0KCiMjIFF1aWNrc3RhcnQgd2l0aCBPbGxhbWEKCkNyZWF0ZSBhIGZpbGUgbmFtZWQg"
    "YE1vZGVsZmlsZWA6CgpgYGBkb2NrZXJmaWxlCkZST00gLi9nZW1tYS0yLTliLWxpdC1kcG8tUTRf"
    "S19NLmdndWYKClBBUkFNRVRFUiB0ZW1wZXJhdHVyZSAwLjY4ClBBUkFNRVRFUiBtaW5fcCAwLjA4"
    "ClBBUkFNRVRFUiB0b3BfazQ0MApQQVJBTUVURVIgcmVwZWF0X3BlbmFsdHkgMS4xNQpQQVJBTUVU"
    "RVIgbnVtX3ByZWRpY3QgMjAwCmBgYAoKQnVpbGQgYW5kIHJ1bjoKYGBgYmFzaAptb2RlbGZpbGU9"
    "TW9kZWxmaWxlCm9sbGFtYSBjcmVhdGUgZ2VtbWEyLWxpdCAtZiBNb2RlbGZpbGUKb2xsYW1hIHJ1"
    "biBnZW1tYTItbGl0CmBgYAoKLS0tCgojIyBQcm9tcHQgVGVtcGxhdGUKCmBgYHRleHQKW1NDRU5F"
    "IFNQSU5FXQpBIHNhaWxvciBvbiBhbiBhYmFuZG9uZWQgZG9jayBzbGlwd2F5IHVuaGl0Y2hlcyBh"
    "IGZyb3plbiBoZW1wIGxpbmUgZnJvbSBhbiBpcm9uIGNsZWF0IHVuZGVyIGEgZnJlZXppbmcgcmFp"
    "biBhdCBsb3cgdGlkZS4gV2l0aCByYXcgaGFuZHMgaGUgd29ya3MgdGhlIHN0aWZmIGZpYmVycyBm"
    "cmVlIGFtaWQgYml0dGVyIHNhbHQgc3ByYXksIG51bWIga251Y2tsZXMsIGFuZCB0aGUgc3RlbmNo"
    "IG9mIHJvdHRlbiBrZWxwLgoKW1BST1NFXQpgYGAK"
)

raw_readme_bytes = base64.b64decode(B64_GGUF_README)

print("Uploading decoded README.md model card...")
api.upload_file(
    path_or_fileobj=raw_readme_bytes,
    path_in_repo="README.md",
    repo_id=GGUF_REPO_ID,
    repo_type="model",
    commit_message="Add GGUF model card with Ollama and LM Studio instructions"
)

print(f"\nSuccessfully published GGUFs to: https://huggingface.co/{GGUF_REPO_ID}")

Creating/verifying Hugging Face repository: czcollier/gemma-2-9b-literary-prose-dpo-GGUF...
Uploading Q4_K_M quantization...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-2-9b-lit-dpo-Q4_K_M.gguf:   0%|          | 21.8MB / 5.76GB            

Validating                    :           |   0%            

Uploading Q8_0 quantization...


... (2 outputs left out)

  ...ma-2-9b-lit-dpo-Q8_0.gguf:   0%|          | 5.92MB / 9.83GB            

Validating                    :           |   0%            

Uploading decoded README.md model card...


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xaa in position 1280: invalid start byte

In [ ]:
import torch

class LongFormLiteraryGenerator:
    def __init__(self, model, tokenizer, device="cuda"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

        # Golden sampling preset
        self.gen_config = {
            "temperature": 0.68,
            "min_p": 0.08,
            "top_k": 40,
            "repetition_penalty": 1.15,
            "do_sample": True,
        }

    def _extract_tail_anchor(self, text: str, max_sentences: int = 2) -> str:
        """Extracts the final 1-2 clean sentences from preceding output."""
        sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
        if not sentences:
            return ""
        tail = ". ".join(sentences[-max_sentences:]) + "."
        return tail

    def generate_beat(self, scene_spine: str, preceding_tail: str = "", max_new_tokens: int = 150) -> str:
        """Generates a single paragraph conditioned on the spine and preceding tail."""
        if preceding_tail:
            prompt = (
                f"[PRECEDING SCENE]\n"
                f"...{preceding_tail}\n\n"
                f"[SCENE SPINE]\n"
                f"{scene_spine.strip()}\n\n"
                f"[PROSE]\n"
            )
        else:
            prompt = (
                f"[SCENE SPINE]\n"
                f"{scene_spine.strip()}\n\n"
                f"[PROSE]\n"
            )

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        prompt_len = inputs["input_ids"].shape[1]

        with torch.inference_mode():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                **self.gen_config
            )

        # Slice cleanly past the prompt tokens
        generated_tokens = output_ids[0][prompt_len:]
        paragraph = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        # Ensure paragraph terminates at a complete sentence boundary
        if "." in paragraph:
            last_period = paragraph.rfind(".")
            paragraph = paragraph[:last_period + 1]

        return paragraph

    def generate_story(self, scene_spines: list[str]) -> str:
        """Executes a multi-beat generation loop over a list of narrative spines."""
        story_paragraphs = []
        preceding_tail = ""

        for idx, spine in enumerate(scene_spines):
            print(f">>> Composing Beat {idx + 1}/{len(scene_spines)}...")

            paragraph = self.generate_beat(
                scene_spine=spine,
                preceding_tail=preceding_tail
            )

            story_paragraphs.append(paragraph)
            # Update the rolling anchor for the next iteration
            preceding_tail = self._extract_tail_anchor(paragraph)

        return "\n\n".join(story_paragraphs)

In [ ]:
story_beats = [
    # Beat 1: Dock slipway
    "A sailor on an abandoned dock slipway unhitches a frozen hemp line from an iron cleat under a freezing rain at low tide. With raw hands he works the stiff fibers free amid bitter salt spray, numb knuckles, and the stench of rotten kelp.",

    # Beat 2: Ascent to shanty
    "He scrambles up the slimy timbers of the breakwater with the coiled hawser dragging behind his boots. He reaches a weather-beaten watchman's shanty where a rusted padlock hangs loose from a rotting pine hasp.",

    # Beat 3: Entering interior
    "He shoulders the swollen door open against a drift of dead leaves and steps inside into the damp gloom. He strikes a sulfur match against his dry thumbnail, illuminating an iron stove coated in orange rust and an empty tin mug on a bench.",

    # Beat 4: Kindling the stove
    "He kneels before the ash pit and feeds dry wood shavings and grease paper into the grate. Blowing gently until a thin orange tongue of flame licks upward, acrid pine smoke billows out to warm his stiff fingers."
]

generator = LongFormLiteraryGenerator(eval_model, tokenizer)
full_story = generator.generate_story(story_beats)

print("\n" + "="*60 + "\n")
print(full_story)

>>> Composing Beat 1/4...
>>> Composing Beat 2/4...
>>> Composing Beat 3/4...
>>> Composing Beat 4/4...


The weather turns bitterly cold with steady sleet that drives straight through his clothes while soaking to the skin in every crevice where it lodges against his flesh like fine needles of ice. He stands upon a slippery plank beside the open sea as icy raindrops strike him full across the face without respite or pause for breath until they sting into sharp pains behind each eye socket. A thin sheet of brine spreads over everything within reach along the deserted pierhead beneath the grey sky; even the air itself tastes salty, leaving his parched throat roughened by gritted sand.

The rain beats down relentlessly against his bare head while he climbs to safety above the floodline on one side and coils away the heavy rope toward dry land on the other. At last he gains foothold atop the battered roof of the old stone lighthouse that once marks this harbour entrance before its lantern 

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

GGUF_REPO_ID = "czcollier/gemma-2-9b-literary-prose-dpo-GGUF"
api = HfApi()

local_readme = Path("GGUF_README.md")

# Write directly to disk using standard multiline blocks
local_readme.write_text(
    """---
language:
- en
license: gemma
base_model: czcollier/gemma-2-9b-literary-prose-dpo
tags:
- gguf
- llama.cpp
- gemma2
- literature
- creative-writing
- dpo
pipeline_tag: text-generation
---

# Gemma-2-9B Literary Prose (DPO) - GGUF

Quantized GGUF binaries of [`czcollier/gemma-2-9b-literary-prose-dpo`](https://huggingface.co/czcollier/gemma-2-9b-literary-prose-dpo) for local inference via **LM Studio**, **Ollama**, and **llama.cpp**.

Fine-tuned on authentic late-19th and early-20th-century literature (*Conrad, Woolf, James, Melville, Fitzgerald, Joyce*) using SFT warm-up followed by low-beta Direct Preference Optimization to suppress conversational AI clichés and output dense, present-tense periodic prose.

---

## Available Files

| Filename | Quant Method | File Size | Description |
| :--- | :--- | :--- | :--- |
| `gemma-2-9b-lit-dpo-Q4_K_M.gguf` | Q4_K_M | ~5.76 GB | Recommended balance of speed, RAM, and prose quality. |
| `gemma-2-9b-lit-dpo-Q8_0.gguf` | Q8_0 | ~9.83 GB | Near-lossless precision matching native BF16. |

---

## Recommended Sampling Configuration

Because Gemma-2 uses logit soft-capping, set your local inference engine to these parameters for optimal syntax stability:

* **Temperature:** `0.68`
* **Min-P:** `0.08`
* **Top-K:** `40`
* **Repetition Penalty:** `1.15`
* **Max Tokens:** `160` - `220`

---

## Quickstart with Ollama

Create a file named `Modelfile`:

```dockerfile
FROM ./gemma-2-9b-lit-dpo-Q4_K_M.gguf

PARAMETER temperature 0.68
PARAMETER min_p 0.08
PARAMETER top_k 40
PARAMETER repeat_penalty 1.15
PARAMETER num_predict 200
```

Build and run:

```Bash
ollama create gemma2-lit -f Modelfile
ollama run gemma2-lit
```

Prompt Template

```Plaintext
[SCENE SPINE]
A sailor on an abandoned dock slipway unhitches a frozen hemp line from an iron cleat under a freezing rain at low tide. With raw hands he works the stiff fibers free amid bitter salt spray, numb knuckles, and the stench of rotten kelp.

[PROSE]
```
""",
encoding="utf-8",
)

print(f"Uploading verified {local_readme} to {GGUF_REPO_ID}...")
api.upload_file(
path_or_fileobj=str(local_readme),
path_in_repo="README.md",
repo_id=GGUF_REPO_ID,
repo_type="model",
commit_message="Add GGUF model card documentation",
)

print(f"Successfully published README to https://huggingface.co/{GGUF_REPO_ID}")

Uploading verified GGUF_README.md to czcollier/gemma-2-9b-literary-prose-dpo-GGUF...
Successfully published README to https://huggingface.co/czcollier/gemma-2-9b-literary-prose-dpo-GGUF


In [ ]:
from datasets import load_dataset
from huggingface_hub import HfApi
from pathlib import Path

DATASET_REPO_ID = "czcollier/literary-prose-dpo-pairs"
LOCAL_JSONL_PATH = "/content/drive/MyDrive/literature_model/dpo_lit_compose_present_deanonymized.jsonl"

# 1. Load the JSONL into a Hugging Face Dataset object
print(f"Loading dataset from {LOCAL_JSONL_PATH}...")
ds = load_dataset("json", data_files={"train": LOCAL_JSONL_PATH})

# 2. Push to Hub (automatically converts to Parquet and enables the Dataset Viewer)
print(f"Pushing dataset to {DATASET_REPO_ID}...")
ds.push_to_hub(
    repo_id=DATASET_REPO_ID,
    private=False,
    commit_message="Upload 966 de-anonymized present-tense literary DPO pairs"
)

# 3. Write and upload the Dataset Card (README.md) without triggering notebook markdown rendering
dataset_readme = Path("DATASET_README.md")
dataset_readme.write_text(
    "\n".join([
        "---",
        "language:",
        "- en",
        "license: cc0-1.0",
        "task_categories:",
        "- text-generation",
        "tags:",
        "- dpo",
        "- preference-dataset",
        "- literature",
        "- creative-writing",
        "- style-transfer",
        "size_categories:",
        "- n<1K",
        "---",
        "",
        "# Literary Prose DPO Pairs (De-anonymized, Present-Tense)",
        "",
        "A curated preference dataset of 966 `(prompt, chosen, rejected)` triplets designed to align language models toward dense, sensory, present-tense literary prose while suppressing conversational AI clichés.",
        "",
        "## Dataset Structure",
        "",
        "- **`prompt`**: A 2-to-3 sentence present-tense narrative scene outline enclosed in `[SCENE SPINE]` and `[PROSE]` tags.",
        "- **`chosen`**: Authentic human-authored literary prose drawn from 12 public-domain Standard Ebooks editions (*Conrad, Woolf, James, Melville, Fitzgerald, Joyce*), scrubbed of proper character names via spaCy and inflected into strict present tense via `lemminflect`.",
        "- **`rejected`**: Matched scenes written in standard conversational AI assistant prose (containing typical RLHF tropes, shallow summaries, and tripartite clichés).",
        "",
        "## License",
        "",
        "Derived from public-domain texts via Standard Ebooks. Released under CC0 1.0 Public Domain Dedication.",
    ]),
    encoding="utf-8"
)

api = HfApi()
api.upload_file(
    path_or_fileobj=str(dataset_readme),
    path_in_repo="README.md",
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    commit_message="Add dataset card documentation"
)

print(f"✓ Dataset live at: https://huggingface.co/datasets/{DATASET_REPO_ID}")

Loading dataset from /content/drive/MyDrive/literature_model/dpo_lit_compose_present_deanonymized.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Pushing dataset to czcollier/literary-prose-dpo-pairs...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpwsqbb29j.parquet    :  56%|#####6    |  526kB /  939kB            

Validating                    :           |   0%            

✓ Dataset live at: https://huggingface.co/datasets/czcollier/literary-prose-dpo-pairs


In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

HF_REPO_ID = "czcollier/gemma-2-9b-literary-prose-dpo"
DATASET_REPO_ID = "czcollier/literary-prose-dpo-pairs"

lines = [
    "---",
    "language:",
    "- en",
    "license: gemma",
    "datasets:",
    f"- {DATASET_REPO_ID}",
    "base_model: google/gemma-2-9b",
    "tags:",
    "- peft",
    "- lora",
    "- dpo",
    "- text-generation",
    "- gemma2",
    "- literature",
    "- creative-writing",
    "pipeline_tag: text-generation",
    "---",
    "",
    "# Gemma-2-9B Literary Prose Adapter (QLoRA + DPO)",
    "",
    "A fine-tuned LoRA adapter for `google/gemma-2-9b` trained to compose dense, sensory, present-tense literary prose in a late-19th / early-20th-century register from compact narrative scene spines.",
    "",
    "Fine-tuned on passages from 12 Standard Ebooks editions (*Conrad, Woolf, James, Melville, Fitzgerald, Joyce*) using a two-stage pipeline (SFT warm-up followed by Direct Preference Optimization). The model suppresses conversational assistant habits and corporate filler in favor of periodic syntax, concrete material nouns, and subordinate clause balance.",
    "",
    "---",
    "",
    "## Model Details",
    "",
    "- **Developed by:** Chris Collier",
    "- **Model Type:** PEFT / LoRA Adapter for Causal LM",
    "- **Base Model:** `google/gemma-2-9b`",
    "- **Architecture:** QLoRA (Rank 16, Alpha 32) targeting all linear attention and MLP projections",
    "- **Training Corpus:** 966 de-anonymized, present-tense narrative pairs (`czcollier/literary-prose-dpo-pairs`)",
    "- **Alignment Technique:** Two-stage (2.5 Epoch SFT Warm-up -> 2.0 Epoch DPO at beta = 0.040)",
    "- **Primary Language:** English",
    "",
    "---",
    "",
    "## Prompt Format",
    "",
    "The adapter expects a continuous, 2-to-3 sentence narrative outline enclosed in `[SCENE SPINE]` and `[PROSE]` delimiters:",
    "",
    "    [SCENE SPINE]",
    "    A sailor on an abandoned dock slipway unhitches a frozen hemp line from an iron cleat under a freezing rain at low tide. With raw hands he works the stiff fibers free amid bitter salt spray, numb knuckles, and the stench of rotten kelp.",
    "",
    "    [PROSE]",
    "",
    "---",
    "",
    "## Recommended Sampling Hyperparameters",
    "",
    "| Parameter | Recommended (Grounded Periodic) | Strict / Minimalist | Range |",
    "| :--- | :--- | :--- | :--- |",
    "| **Temperature** | **0.68** | 0.45 | 0.45 - 0.78 |",
    "| **min_p** | **0.08** | 0.05 | 0.05 - 0.10 |",
    "| **top_k** | **40** | 40 | 30 - 50 |",
    "| **repetition_penalty** | **1.12** | 1.10 | 1.05 - 1.15 |",
    "| **max_new_tokens** | **160** | 140 | 120 - 220 |",
    "",
    "---",
    "",
    "## Quickstart (Transformers + PEFT)",
    "",
    "    import torch",
    "    from transformers import AutoModelForCausalLM, AutoTokenizer",
    "    from peft import PeftModel",
    "",
    '    BASE_MODEL_ID = "google/gemma-2-9b"',
    '    ADAPTER_ID = "czcollier/gemma-2-9b-literary-prose-dpo"',
    "",
    "    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)",
    "    base_model = AutoModelForCausalLM.from_pretrained(",
    "        BASE_MODEL_ID,",
    "        torch_dtype=torch.bfloat16,",
    '        device_map="auto",',
    '        attn_implementation="eager",',
    "    )",
    "",
    "    model = PeftModel.from_pretrained(base_model, ADAPTER_ID)",
    "    model = model.merge_and_unload()",
    "    model.eval()",
    "",
    '    prompt = "[SCENE SPINE]\\nA sailor on an abandoned dock slipway unhitches a frozen hemp line from an iron cleat under a freezing rain at low tide. With raw hands he works the stiff fibers free amid bitter salt spray, numb knuckles, and the stench of rotten kelp.\\n\\n[PROSE]\\n"',
    '    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")',
    "",
    "    with torch.inference_mode():",
    "        output_ids = model.generate(",
    "            **inputs,",
    "            max_new_tokens=160,",
    "            temperature=0.68,",
    "            min_p=0.08,",
    "            top_k=40,",
    "            repetition_penalty=1.12,",
    "            do_sample=True,",
    "            pad_token_id=tokenizer.pad_token_id,",
    "            eos_token_id=tokenizer.eos_token_id,",
    "        )",
    "",
    "    print(tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip())",
    "",
    "---",
    "",
    "## Sample Output (Grounded Periodic, T = 0.68)",
    "",
    "**Prompt Spine:**",
    "*A sailor on an abandoned dock slipway unhitches a frozen hemp line from an iron cleat under a freezing rain at low tide. With raw hands he works the stiff fibers free amid bitter salt spray, numb knuckles, and the stench of rotten kelp.*",
    "",
    "**Output:**",
    '> *"The wet rope clings to its rusted spike as if glued fast by ice; yet after long labor the soaked strand yields slowly inch by inch until it slips clear with a sharp jerk. The line remains coiled upon the planks where men once cast off their hawsers before they climb aboard ship in fair weather or foul, but now no vessel rides moored against this pierhead beneath the driving sleet that lashes across the exposed cobbles."*',
    "",
    "---",
    "",
    "## Training Hyperparameters",
    "",
    "- **Stage 1 (SFT Warm-up):** 2.5 epochs (~600 steps), effective batch size 4, LR = 3.5e-5 (cosine schedule, 8% warmup)",
    "- **Stage 2 (DPO Alignment):** 2.0 epochs (~480 steps), effective batch size 4, beta = 0.040, LR = 4.0e-7",
    "- **Hardware:** 1x NVIDIA A100",
]

readme_path = Path("CLEAN_MODEL_CARD.md")
readme_path.write_text("\n".join(lines), encoding="utf-8")

api = HfApi()
api.upload_file(
    path_or_fileobj=str(readme_path),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Fix formatting typos in quickstart tag, LaTeX, and update showcase sample"
)
print("✓ Model card updated cleanly.")

✓ Model card updated cleanly.
